# Overview of the Encoder Architecture

![encoder_overview](../resources/v0_7/encoder_overview.png)

The encoder architecture is transformer-based with linear attention, SwiGLU and RMSNorm. As part of the encoder we also have a second masked predictor that processes the output of the masked context encoder during training. This notebook will walk through each layer so that the reader can build an intuition for what each layer is doing to the data. To this extent, you'll see that we set the layer initializations and numbers to ones where you can hand calculate if you need to follow a layer better. 

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import math

## Data Prep

We'll start with a simple data prep. Here we'll use a small batch of 2 samples, each with 9 gene expression counts. 

In [2]:
batch = 2 # Batch
num_genes = 9 # context, aka num of genes


SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
x_input = torch.from_numpy(np.round(np.random.uniform(1, 5, size=(batch, num_genes)), 0)).float() # [batch, num_genes]
total_counts = torch.from_numpy(np.random.randint(1, 10, size=(batch))).float()# [batch]

x_input.shape, x_input, total_counts.shape, total_counts

(torch.Size([2, 9]),
 tensor([[2., 2., 2., 3., 2., 3., 2., 5., 4.],
         [1., 3., 4., 2., 5., 3., 4., 4., 2.]]),
 torch.Size([2]),
 tensor([4., 4.]))

## Unknown Gene Handling
Our datasets don't always have every gene. Because of this, we need to have a way of handling when a gene isn't in our training data differently than when a gene is 0 count or masked. To handle this, during data prep we create a mask to flag which of the 10k genes a sample has. We'll for now stage a mask and show how each sample can have a different mask. 

In [4]:
unknown_mask = torch.tensor([
    [False, True, False, False, False, False, False, False, True],
    [False, False, True, False, False, False, False, False, False]
])
unknown_mask.shape, unknown_mask

(torch.Size([2, 9]),
 tensor([[False,  True, False, False, False, False, False, False,  True],
         [False, False,  True, False, False, False, False, False, False]]))

## Data Masking 
(only done for the Context/Student) The self-supervised learning is done via masked prediction where we create a mask to make the target prediction task harder. During loss, we primarily start by evaluating the masked positions and slowly add in the rest (excluding unknown). We'll first generate a random distribution and then everything below our target masking threshold will be masked. For the sake of the demonstration I'll use a lower masking ratio than our model.

We apply the masking twice, before our forward pass and after the input projection.  We do this first masking of the raw expression values before they enter the encoder to prevent the expression value from influencing the FiLM conditioning (the Fourier features and gamma/beta) at masked positions. Without this, the nonlinear FiLM path would be influenced by the masked positions even though the position gets replaced later. The unknown positions are different as they don't contain real info so there's no information to leak. 

In [5]:
mask_ratio = 0.4 
rand = torch.rand(batch, num_genes)
rand

tensor([[0.0783, 0.4956, 0.6231, 0.4224, 0.2004, 0.0287, 0.5851, 0.6967, 0.1761],
        [0.2595, 0.7086, 0.5809, 0.0574, 0.7669, 0.8778, 0.2434, 0.6005, 0.7079]])

In [6]:
mask_idx = rand < mask_ratio #use probabilistic masking. it's not perfect but will work well over a large training run
mask_idx

tensor([[ True, False, False, False,  True,  True, False, False,  True],
        [ True, False, False,  True, False, False,  True, False, False]])

In [7]:
x_values = x_input
x_values[mask_idx] = 0.0
x_values

tensor([[0., 2., 2., 3., 0., 0., 2., 5., 0.],
        [0., 3., 4., 0., 5., 3., 0., 4., 2.]])

## Forward Pass

We start by inserting in a channels dimension. Right now we just have 1 value per gene, but we'll represent each gene with many dimensions `embed_dim` to let the model learn different combinations of gene importance. We'll also use multiple `heads` which is basically a grouping of the embedding dimension channels so that combinations of them can learn different complex topics. 

In [8]:
embed_dim = 6
heads = 2

**Insert in the channels dimension**

In [9]:
x = x_values.unsqueeze(-1)
x.shape, x

(torch.Size([2, 9, 1]),
 tensor([[[0.],
          [2.],
          [2.],
          [3.],
          [0.],
          [0.],
          [2.],
          [5.],
          [0.]],
 
         [[0.],
          [3.],
          [4.],
          [0.],
          [5.],
          [3.],
          [0.],
          [4.],
          [2.]]]))

### Fourier FiLM gene encoding

Instead of just relying on gene expression counts, we want to do a Fourier FiLM based projection of the gene expression counts with learnable controls on the projection. 

![full_model_overview](../resources/v0_7/fourier_film_encoder.png)

We do this since we know that in biology, expression is typically non-linear where you have different expression plateaus including fully on or off. In this projection expression values are first scaled by learned gene embeddings and a scaler, then a random Fourier feature network generates expression-dependent FiLM parameters (gamma, beta) to further modulate the result based on expression level. The two paths give the model both a linear signal (expression * embedding) and a nonlinear one (Fourier features -> FiLM), combined into the final gene representation. The goal is to apply the following

$h_g = \alpha x_g e_g \odot (1 + \gamma_g) + \beta_g$

where

$[\gamma_g, \beta_g] = \mathrm{MLP}(\phi(\alpha' x_g))$

$x_g$ is the expression value for gene $g$ while $e_g$ is the learned gene embedding, $\alpha, \alpha'$ are learned scalars and $\phi$ is a random Fourier feature mapping 

While we use a multilayer perceptron (MLP), the FiLM portion is specifically the $\mathbf{h}_g = x * (1 + \gamma) + \beta$ pattern. This becomes an affine transformation where gamma and beta are conditioned on some input. 

You might notice that if we simply did this, we'd lose the gene identity for true zero-count genes in our data.  After we calculate the projection, we then add back the gene identity as a residual so that after the FiLM modulation, each gene that exists in the dataset retains a baseline gene identity signal regardless of expression level.  This allows the model to build different influences on the gene embeddings for "gene exists" and "gene expressed by this much" 

#### $\alpha$ Gene Expression Count Scaling

We'll start with our gene expression count scaler. This will be a single value that we'll multiply against our scaled gene expression count embeddings. We use a linear layer here so that backprop can update this scaler. We'll first setup the learned scaler and multiply it by expression counts, after which we'll then use that to scale our gene embeddings. We'll initialize this scaler to `1.5` so you'll see that our initial expression values grow.

In [10]:
expr_scaler = nn.Linear(1, 1, bias=False)
nn.init.constant_(expr_scaler.weight, 1.5)
expr_scaler.weight

Parameter containing:
tensor([[1.5000]], requires_grad=True)

In [11]:
scaled_x = expr_scaler(x)
scaled_x.shape, scaled_x

(torch.Size([2, 9, 1]),
 tensor([[[0.0000],
          [3.0000],
          [3.0000],
          [4.5000],
          [0.0000],
          [0.0000],
          [3.0000],
          [7.5000],
          [0.0000]],
 
         [[0.0000],
          [4.5000],
          [6.0000],
          [0.0000],
          [7.5000],
          [4.5000],
          [0.0000],
          [6.0000],
          [3.0000]]], grad_fn=<UnsafeViewBackward0>))

#### $\mathbf{e}_g$ Gene Embeddings

Now we'll initialize a representation of the gene embeddings and then scale them based on our scaled expression counts. You'll see that the masking extends across all embedding channels. Also, since we used an initialization where each channel has the same value, you'll see that the multiplication creates consistent channel entries per gene

In [12]:
# creates an incremental weight for easier following
vs, d = num_genes, embed_dim
rows = torch.arange(vs).unsqueeze(1)
cols = torch.full((d,), 1.0).unsqueeze(0)
pattern = 0.1*(rows + cols)  

gene_embeddings = nn.Parameter(pattern)
gene_embeddings

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000],
        [0.7000, 0.7000, 0.7000, 0.7000, 0.7000, 0.7000],
        [0.8000, 0.8000, 0.8000, 0.8000, 0.8000, 0.8000],
        [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000]], requires_grad=True)

In [13]:
scaled_x = gene_embeddings.unsqueeze(0) * scaled_x
scaled_x

tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000],
         [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000],
         [1.8000, 1.8000, 1.8000, 1.8000, 1.8000, 1.8000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [2.1000, 2.1000, 2.1000, 2.1000, 2.1000, 2.1000],
         [6.0000, 6.0000, 6.0000, 6.0000, 6.0000, 6.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000],
         [1.8000, 1.8000, 1.8000, 1.8000, 1.8000, 1.8000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [3.7500, 3.7500, 3.7500, 3.7500, 3.7500, 3.7500],
         [2.7000, 2.7000, 2.7000, 2.7000, 2.7000, 2.7000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [4.8000, 4.8000, 4.8000, 4.8000, 4.8000, 4.80

#### $\alpha'$ Fourier Scaler 

Now we move on to the Fourier half of the calculation. We'll start by calculating the scaler. Again we use a linear here so that it represents a learnable parameter that will drift during backprop. To make sure the values diverge from the expression scaler, I'll use a different value `0.5`. When applied, you can see our initial expression values are cut in half. 

In [14]:
fourier_input_scaler = nn.Linear(1, 1, bias=False)
nn.init.constant_(fourier_input_scaler.weight, 0.5)
fourier_input_scaler.weight

Parameter containing:
tensor([[0.5000]], requires_grad=True)

In [15]:
fourier_x = fourier_input_scaler(x)
fourier_x.shape, fourier_x

(torch.Size([2, 9, 1]),
 tensor([[[0.0000],
          [1.0000],
          [1.0000],
          [1.5000],
          [0.0000],
          [0.0000],
          [1.0000],
          [2.5000],
          [0.0000]],
 
         [[0.0000],
          [1.5000],
          [2.0000],
          [0.0000],
          [2.5000],
          [1.5000],
          [0.0000],
          [2.0000],
          [1.0000]]], grad_fn=<UnsafeViewBackward0>))

#### $\phi$ Fourier Projection  
Now we'll do the Fourier projection. This step projects the scaled expression counts through a fixed random matrix `fp_scaler`. This random matrix is half the size since we take the sin and cos of the result and then concatenate them together to produce a full-dimensional embedding. 

This step maps the continuous expression level into a rich high-dimensional representation where nearby values have similar features allowing the model to learn representations from the different plateaus of expression levels. The `gaussian_scale` scaler is a tunable hyperparameter that can quickly improve and degrade model performance.

Since we're working with sine/cosine we have to think of the periodicity. We first will take our expression and spread them across a full sin/cosine cycle, which, if you remember your trig, is $2\pi$. This normalizes the input so that a unit change in the input corresponds to one full cycle of sin/cos. If we did not do this, the random projection matrix alone controls the frequency, increasing the fragility of the projection.

**Numeric computing** one thing to remember is that we're working with fixed precision. Because of this, when we use irrational numbers like $\pi$, we must store a numeric representation meaning only a certain precision is stored. This means that while we should expect integer representations for multiples of $\pi$, we actually sometimes get infinitesimals instead. While it's not the most precise, this randomly introduced noise will just act as a mini-bias to our model. Take the below example, for `sin` we should see `[0,1,0,-1,0]` and for `cos` we should see `[1,0,-1,0,1]` but instead we see some values not quite there. Keep this in mind as we'll see this numeric computing error introduced in our notebook. 

In [16]:
examp_array = torch.tensor([0,0.25,0.5,0.75,1]).float()
torch.sin(examp_array*2*np.pi), torch.cos(examp_array*2*np.pi)

(tensor([ 0.0000e+00,  1.0000e+00, -8.7423e-08, -1.0000e+00,  1.7485e-07]),
 tensor([ 1.0000e+00, -4.3711e-08, -1.0000e+00,  1.1925e-08,  1.0000e+00]))

In [17]:
x_fp = (2 * np.pi * fourier_x)
x_fp

tensor([[[ 0.0000],
         [ 6.2832],
         [ 6.2832],
         [ 9.4248],
         [ 0.0000],
         [ 0.0000],
         [ 6.2832],
         [15.7080],
         [ 0.0000]],

        [[ 0.0000],
         [ 9.4248],
         [12.5664],
         [ 0.0000],
         [15.7080],
         [ 9.4248],
         [ 0.0000],
         [12.5664],
         [ 6.2832]]], grad_fn=<MulBackward0>)

Now that we've scaled our expression counts by $2\pi$, we'll inject half of the channels. As part of this injection, we use random initiated noise for each channel and a tunable hyperparameter scaler `gaussian_scale` to increment the noise. For learning, we'll keep our initialization preset.

That said, you'll see that we do not use gradients as this is not learnable. Our goal with the random Fourier features is that a fixed random projection is theoretically sufficient to approximate a shift-invariant kernel (we care about differences in expression values, not the actual numeric number). If we made it learnable, the network would likely collapse the frequencies to overfit or degenerate, defeating the purpose of providing a diverse multi-frequency basis. The nonlinear expressiveness of this layer comes downstream with the film generation MLP that processes the Fourier features. That layer is learnable.

In [18]:
gaussian_scale = 2.0
fp_scaler = nn.Parameter(torch.tensor([[0.5,1.0,1.5]])* gaussian_scale, requires_grad=False)
fp_scaler

Parameter containing:
tensor([[1., 2., 3.]])

In [19]:
x_fp = x_fp @ fp_scaler
x_fp.shape, x_fp

(torch.Size([2, 9, 3]),
 tensor([[[ 0.0000,  0.0000,  0.0000],
          [ 6.2832, 12.5664, 18.8496],
          [ 6.2832, 12.5664, 18.8496],
          [ 9.4248, 18.8496, 28.2743],
          [ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [ 6.2832, 12.5664, 18.8496],
          [15.7080, 31.4159, 47.1239],
          [ 0.0000,  0.0000,  0.0000]],
 
         [[ 0.0000,  0.0000,  0.0000],
          [ 9.4248, 18.8496, 28.2743],
          [12.5664, 25.1327, 37.6991],
          [ 0.0000,  0.0000,  0.0000],
          [15.7080, 31.4159, 47.1239],
          [ 9.4248, 18.8496, 28.2743],
          [ 0.0000,  0.0000,  0.0000],
          [12.5664, 25.1327, 37.6991],
          [ 6.2832, 12.5664, 18.8496]]], grad_fn=<UnsafeViewBackward0>))

**Sin/Cos**

Now we're ready for our radial extraction to take the sine and cosine. As a reminder, because we're dealing with numeric computing, instead of nice clean integers we'll see that we have infinitesimals introduced. Interestingly, this is more prevalent in sine vs cosine bringing us back up to our embedding dimension size. 

In [20]:
x_fp_sin = torch.sin(x_fp)
x_fp_sin

tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08],
         [-6.7553e-07,  1.3511e-06, -3.9339e-06],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00]],

        [[ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 3.4969e-07,  6.9938e-07,  9.5399e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [-6.7553e-07,  1.3511e-06, -3.9339e-06],
         [-2.3850e-08,  4.7700e-08, -7.1549e-08],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 3.4969e-07,  6.9938e-07,  9.5399e-08],
         [ 1.7485e-07,  3.4969e-07,  4.7700e-08]]], grad_fn=<SinBackward0>)

In [21]:
x_fp_cos = torch.cos(x_fp)
x_fp_cos

tensor([[[ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.]],

        [[ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [-1.,  1., -1.],
         [-1.,  1., -1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.],
         [ 1.,  1.,  1.]]], grad_fn=<CosBackward0>)

In [22]:
fourier_x = torch.cat([x_fp_sin, x_fp_cos], dim=-1)
fourier_x.shape, fourier_x

(torch.Size([2, 9, 6]),
 tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [-2.3850e-08,  4.7700e-08, -7.1549e-08, -1.0000e+00,  1.0000e+00,
           -1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [ 1.7485e-07,  3.4969e-07,  4.7700e-08,  1.0000e+00,  1.0000e+00,
            1.0000e+00],
          [-6.7553e-07,  1.3511e-06, -3.9339e-06, -1.0000e+00,  1.0000e+00,
           -1.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  1.0000e+00,
            1.0000e+00]],
 
         [[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  

#### $\mathrm{MLP}$ Multilayer perceptron  

Now we're ready to add the nonlinear expressiveness for the film generator by using a MLP. The MLP will provide a learnable linear layer, a nonlinearity, and a final upward projection into a doubling to create our $\gamma$ and $\beta$ for our FiLM calculation. These learnable layers are what allow the model to decide how much and which parts of the Fourier Projection we want to include with our initial scaled expression. 

**MLP - linear 1** We'll first start with a single linear layer that allows the model to decide how much each channel learned should interact with the other. Recall that half our channels are sine, and half are cosine, so this allows the model to mix the two radial projections. 

*You'll notice that near 0 values here are washed out*

In [23]:
mlp_l1 = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(mlp_l1.weight, .25)
nn.init.constant_(mlp_l1.bias, 0.0)
mlp_l1.weight

Parameter containing:
tensor([[0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500]], requires_grad=True)

In [24]:
fourier_x = mlp_l1(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 9, 6]),
 tensor([[[ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500]],
 
         [[ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [ 0.7500,  0.7500,  0.7500,  0.7500,  0.7500,  0.7500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0.2500, -0.2500],
          [-0.2500, -0.2500, -0.2500, -0.2500, -0

**MLP - GELU nonlinearity** Now we're ready for our non-linearity. The [GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html) function is approximately linear above 1 and pulls most values below -2 to 0. Between -2 and 0, most values are pulled closer to 0 and there's a slight non-linearity between 0 and 1. Since we used sine/cosine values, most of our values at this point should be between 0 and 1 so we'll benefit from the non-linear portion with a cap on highly negative values. 

In [25]:
mlp_gelu = nn.GELU()

fourier_x = mlp_gelu(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 9, 6]),
 tensor([[[ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800]],
 
         [[ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [ 0.5800,  0.5800,  0.5800,  0.5800,  0.5800,  0.5800],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0.1003, -0.1003],
          [-0.1003, -0.1003, -0.1003, -0.1003, -0

**MLP - linear 2** Now we'll scale up to 2x our embedding size since we need to provide values for our two variables in our FiLM projection. We use a learnable linear scale up layer so that the model can learn how to split values across the two variables. For our initialization we'll increment the second half to be twice the first half to show the differences. 

In [26]:
mlp_l2 = nn.Linear(embed_dim, embed_dim*2)
nn.init.constant_(mlp_l2.weight[:embed_dim, :], 0.5)
nn.init.constant_(mlp_l2.weight[embed_dim:, :], 1.0)
nn.init.constant_(mlp_l2.bias, 0.0)
mlp_l2.weight.shape, mlp_l2.weight

(torch.Size([12, 6]),
 Parameter containing:
 tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]], requires_grad=True))

In [27]:
fourier_x = mlp_l2(fourier_x)
fourier_x.shape, fourier_x

(torch.Size([2, 9, 12]),
 tensor([[[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.6019,
           -0.6019, -0.6019, -0.6019, -0.6019, -0.6019],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  3.4802,
            3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0

#### $\gamma_g, \beta_g$ FiLM components 
Now that we have the MLP output, we're ready to build our FiLM variables. Recall that for FiLM we calculate $\mathbf{h}_g = x * (1 + \gamma) + \beta$ where $x$ will be the weighted scaler of the expression counts. To get $\gamma$ and $\beta$ we simply split the output of the MLP. Recall that we built it so that half of the MLP output was doubled so when we split, we should see that $\beta$ is double $\gamma$.

In [28]:
gamma, beta = torch.chunk(fourier_x, 2, dim=-1)
gamma.shape, gamma, beta.shape, beta

(torch.Size([2, 9, 6]),
 tensor([[[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401]],
 
         [[ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [ 1.7401,  1.7401,  1.7401,  1.7401,  1.7401,  1.7401],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0.3010, -0.3010],
          [-0.3010, -0.3010, -0.3010, -0.3010, -0

#### $\mathbf{h}_g$ FiLM based gene expression representation 

Now we're ready for the final FiLM calculation. FiLM can be thought of as summing two weighted parts, in our case a scaled version of the expression counts and a radial projection of the expression counts. What's interesting is that the FiLM formula pushes the radial projection both as a scaler to the initial counts and a bias similar to $x = mx + b$. Ultimately we've given the model the ability to learn how to upscale the original counts, shift the counts based on the radial projection, and add/subtract the counts based on the radial projection. All of this allows the model to learn a more complex landscape to scale gene embeddings by the expression beyond the typical linear scaling where 2 expression counts mean double 1 expression. 

In [29]:
x = scaled_x * (1.0 + gamma) + beta

x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 5.1242,  5.1242,  5.1242,  5.1242,  5.1242,  5.1242],
          [ 5.9463,  5.9463,  5.9463,  5.9463,  5.9463,  5.9463],
          [ 0.6563,  0.6563,  0.6563,  0.6563,  0.6563,  0.6563],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 9.2344,  9.2344,  9.2344,  9.2344,  9.2344,  9.2344],
          [ 3.5922,  3.5922,  3.5922,  3.5922,  3.5922,  3.5922],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802]],
 
         [[ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 0.0272,  0.0272,  0.0272,  0.0272,  0.0272,  0.0272],
          [ 8.4123,  8.4123,  8.4123,  8.4123,  8.4123,  8.4123],
          [ 3.4802,  3.4802,  3.4802,  3.4802,  3.4802,  3.4802],
          [ 2.0194,  2.0194,  2.0194,  2.0194,  2.0194,  2.0194],
          [ 1.2854,  1.2854,  1.2854,  1.2854,  1

#### Residual Gene Embedding

Now we'll add in our gene embedding back to the expression-based FiLM projection value. As mentioned before, this ensures that the model learns the difference of gene exists and doesn't exist.  If we just let the 0 count project through, we'd collapse all the 0 count genes down to a single representation to the model confusing it further. 

In [30]:
x = gene_embeddings.unsqueeze(0) + x
x

tensor([[[ 3.5802,  3.5802,  3.5802,  3.5802,  3.5802,  3.5802],
         [ 5.3242,  5.3242,  5.3242,  5.3242,  5.3242,  5.3242],
         [ 6.2463,  6.2463,  6.2463,  6.2463,  6.2463,  6.2463],
         [ 1.0563,  1.0563,  1.0563,  1.0563,  1.0563,  1.0563],
         [ 3.9802,  3.9802,  3.9802,  3.9802,  3.9802,  3.9802],
         [ 4.0802,  4.0802,  4.0802,  4.0802,  4.0802,  4.0802],
         [ 9.9344,  9.9344,  9.9344,  9.9344,  9.9344,  9.9344],
         [ 4.3922,  4.3922,  4.3922,  4.3922,  4.3922,  4.3922],
         [ 4.3802,  4.3802,  4.3802,  4.3802,  4.3802,  4.3802]],

        [[ 3.5802,  3.5802,  3.5802,  3.5802,  3.5802,  3.5802],
         [ 0.2272,  0.2272,  0.2272,  0.2272,  0.2272,  0.2272],
         [ 8.7123,  8.7123,  8.7123,  8.7123,  8.7123,  8.7123],
         [ 3.8802,  3.8802,  3.8802,  3.8802,  3.8802,  3.8802],
         [ 2.5194,  2.5194,  2.5194,  2.5194,  2.5194,  2.5194],
         [ 1.8854,  1.8854,  1.8854,  1.8854,  1.8854,  1.8854],
         [ 4.1802,  4.1

#### Remask with learned mask

Now we need to reintroduce the masking back but, instead of a 0 value, we want to actually let the model learn a mask token. Previously we ensured the expression value were hidden for the expression count projection, while this time we create a learnable mask signal for the positions to tell the model to fill them in. We learn a mask token because the model needs to distinguish "this gene is masked and I need to predict it" from "this gene has zero expression." If the mask token were fixed (e.g. all zeros), it would be indistinguishable from a zero-expression gene's representation after the Fourier FiLM encoding. A learned token lets the model settle on a representation that optimally signals "predict me" to the downstream transformer blocks. Ultimately we're reapplying the masking at this point mainly so that we do our expression encoding cleanly first, and then mask after. Since our mask token is learnable, instead of a common `-1` hardcode, the initialization in the model will use random learnable values. What we'll do in our example is use `-11` so it really sticks out. These values will change during backprop as the model learns what a good token is to prompt it that it needs replacement. 

*As a reminder, masking only happens on the context encoder, and not the target encoder during our model's forward pass*

In [31]:
mask_token = nn.Parameter(torch.randn(embed_dim) * 0.02)
nn.init.constant_(mask_token, -11)
mask_token

Parameter containing:
tensor([-11., -11., -11., -11., -11., -11.], requires_grad=True)

In [32]:
x = torch.where(mask_idx.unsqueeze(-1), mask_token, x)
x

tensor([[[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  5.3242,   5.3242,   5.3242,   5.3242,   5.3242,   5.3242],
         [  6.2463,   6.2463,   6.2463,   6.2463,   6.2463,   6.2463],
         [  1.0563,   1.0563,   1.0563,   1.0563,   1.0563,   1.0563],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  9.9344,   9.9344,   9.9344,   9.9344,   9.9344,   9.9344],
         [  4.3922,   4.3922,   4.3922,   4.3922,   4.3922,   4.3922],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000]],

        [[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  0.2272,   0.2272,   0.2272,   0.2272,   0.2272,   0.2272],
         [  8.7123,   8.7123,   8.7123,   8.7123,   8.7123,   8.7123],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  2.5194,   2.5194,   2.5194,   2.5194,   2.5194,   2.5194],
    

#### Mask unknown genes with learnable unknown token

Now we need to reset the unknown gene positions with a learnable unknown token.  This will signal to the model that these positions aren't part of the reconstruction loss and we don't actually know what they are in reality. This helps preserve their identity while giving the model a way to deprioritize them during the learning.  Since our unknown token is learnable the initialization in the model will use random learnable values. What we'll do in our example is use `-5` so it's different than the mask but still sticks out. These values will change during backprop as the model learns what a good token is to prompt it that it needs replacement. 

*As a reminder, unknown masking happens in both the student and teacher encoders*

In [33]:
unknown_token = nn.Parameter(torch.randn(embed_dim) * 0.02)
nn.init.constant_(unknown_token, -5)
unknown_token

Parameter containing:
tensor([-5., -5., -5., -5., -5., -5.], requires_grad=True)

In [34]:
x = torch.where(unknown_mask.unsqueeze(-1), unknown_token, x)
x

tensor([[[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [ -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000],
         [  6.2463,   6.2463,   6.2463,   6.2463,   6.2463,   6.2463],
         [  1.0563,   1.0563,   1.0563,   1.0563,   1.0563,   1.0563],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  9.9344,   9.9344,   9.9344,   9.9344,   9.9344,   9.9344],
         [  4.3922,   4.3922,   4.3922,   4.3922,   4.3922,   4.3922],
         [ -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000]],

        [[-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  0.2272,   0.2272,   0.2272,   0.2272,   0.2272,   0.2272],
         [ -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000,  -5.0000],
         [-11.0000, -11.0000, -11.0000, -11.0000, -11.0000, -11.0000],
         [  2.5194,   2.5194,   2.5194,   2.5194,   2.5194,   2.5194],
    

### Total count injection

Now we'll want to add the total count in. In our data prep, we normalize all cell expression counts to the same total count. This is great since Perturb-seq is relativistic in its data, but this normalization has one drawback: cells with abnormally high or low expression totals compared to their peers lose the signal. In particular the abnormally low is our biggest concern as a perturbation that makes a cell barely viable may have very low across the board expression that gets amplified when you bring it up to our normalized count. To avoid this we add in a learnable weight to the total expression count so that the model can learn expression levels.

The total count is multiplied across the embedding dimensions with a learnable weight and then summed to our gene expression projections. This allows the total count to act almost like a bias term. 

We start by taking our total count, injecting the embedding dimensions, and then shaping it to match our embedding counts. 

In [35]:
x_total_ct = total_counts.unsqueeze(-1)
x_total_ct.shape, x_total_ct

(torch.Size([2, 1]),
 tensor([[4.],
         [4.]]))

In [36]:
total_count_proj = nn.Linear(1, embed_dim)
nn.init.constant_(total_count_proj.weight, 0.1)
nn.init.zeros_(total_count_proj.bias)
total_count_proj.weight

Parameter containing:
tensor([[0.1000],
        [0.1000],
        [0.1000],
        [0.1000],
        [0.1000],
        [0.1000]], requires_grad=True)

In [37]:
x_total_ct = total_count_proj(x_total_ct)
x_total_ct = x_total_ct.unsqueeze(1)
x_total_ct.shape, x_total_ct

(torch.Size([2, 1, 6]),
 tensor([[[0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000]],
 
         [[0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000]]],
        grad_fn=<UnsqueezeBackward0>))

### Unified representation of the cell state

Now that we have an embedding representation of both the gene expression and total count, we're ready to sum them for a single representation of the cell state. Now it will be ready for a cell state block

In [38]:
x = x + x_total_ct
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-10.6000, -10.6000, -10.6000, -10.6000, -10.6000, -10.6000],
          [ -4.6000,  -4.6000,  -4.6000,  -4.6000,  -4.6000,  -4.6000],
          [  6.6463,   6.6463,   6.6463,   6.6463,   6.6463,   6.6463],
          [  1.4563,   1.4563,   1.4563,   1.4563,   1.4563,   1.4563],
          [-10.6000, -10.6000, -10.6000, -10.6000, -10.6000, -10.6000],
          [-10.6000, -10.6000, -10.6000, -10.6000, -10.6000, -10.6000],
          [ 10.3344,  10.3344,  10.3344,  10.3344,  10.3344,  10.3344],
          [  4.7922,   4.7922,   4.7922,   4.7922,   4.7922,   4.7922],
          [ -4.6000,  -4.6000,  -4.6000,  -4.6000,  -4.6000,  -4.6000]],
 
         [[-10.6000, -10.6000, -10.6000, -10.6000, -10.6000, -10.6000],
          [  0.6272,   0.6272,   0.6272,   0.6272,   0.6272,   0.6272],
          [ -4.6000,  -4.6000,  -4.6000,  -4.6000,  -4.6000,  -4.6000],
          [-10.6000, -10.6000, -10.6000, -10.6000, -10.6000, -10.6000],
          [  2.9194,   2.9194,   2.91

### Transformer Block 
The transformer layer in our model consists of 4 layers:
1. RMS normalization
2. Gated linear self-attention
3. RMS normalization
4. SwiGLU

Residual connections provide gradient bypassing around the attention and SwiGLU layers. This set of 4 units is repeated based on how many layers are configured. 

#### RMSNorm 1

For our modern transformer, we use root mean square normalization, or RMSNorm. RMSNorm calculates the following: 
$$
y = \frac{x}{\sqrt{\frac{1}{n} \sum_{i=1}^{n} x_i^2 + \epsilon}} \cdot \gamma
$$

The main reason we use RMSNorm is that it executes faster and uses less memory than standard layer normalization. This efficiency is achieved by entirely removing the mean-centering calculation, which reduces the total number of arithmetic operations and hardware synchronization steps. Recall that normalization is primarily used for large scale training stability (preventing gradient explosion/vanishing). For deep architectures like ours, the mean of the pre-activation inputs naturally stays close to zero during training so removing the mean-centering operation preserves the critical variance-bounding effect. Also the model learns to absorb any minor activation shifts into the subsequent linear weights or the learned affine parameters. Because of this, we're able to use a more efficient normalization.

Since we reuse RMSNorm 3 times, we'll create a class for it. In the class you can see that we split out the calculation, first limiting the precision of X, then computing $\frac{x}{\sqrt{\frac{1}{n} \sum_{i=1}^{n} x_i^2 + \epsilon}}$, and finally adding the per-channel weights $\cdot \gamma$.

Since our entries have identical values across all channels for a gene, the gene becomes uniform +/-1 depending on the sign (the value is the average so it becomes 1)

In [39]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        x_fp32 = x.float()
        norm = x_fp32 * torch.rsqrt(x_fp32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight).type_as(x)

In [40]:
rms1 = RMSNorm(embed_dim)

rms1.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [41]:
x_norm = rms1(x)
x_norm.shape, x_norm

(torch.Size([2, 9, 6]),
 tensor([[[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000]],
 
         [[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000,  1.0000,  1

#### Multi-Headed Gated Linear Attention
Since our current token size is 10000 and we know it's going to grow to 20,000+ when we add all genes and other cell state representations, we want to avoid building the attention matrix in memory. To avoid this we chose linear attention. Linear attention foregoes materializing the attention matrix. In the encoder, we do self attention allowing each gene to build a relationship with any other gene. Our attention will be multi-headed, meaning we'll split up our embedding space across the heads allowing them each to learn different complex representations. Finally we'll also add gating at the end to allow the model to determine how much the attention should impact each specific gene. Linear attention still incorporates the query, key, and value, but removes the need for softmax. We calculate

$$\begin{aligned}
\text{elu}(x) &= \begin{cases} x & \text{if } x > 0 \\ \alpha (e^x - 1) & \text{if } x \le 0 \end{cases} \\
\\
Q &= \text{elu}(x W_q^\top + b_q) + 1.0 \\
K &= \text{elu}(x W_k^\top + b_k) + 1.0 \\
V &= x W_v^\top + b_v \\
\\
\text{Attn} &= \frac{Q K^\top V}{Q (\sum_{j=1}^T K_j)^\top + \epsilon} \\
\text{Gate} &= \sigma(x W_{gate}^\top + b_{gate}) \\
\\
y &= \left( \text{Gate} \odot \text{Attn} \right) W_c^\top + b_c
\end{aligned}$$

In [42]:
B, T_q, C = x_norm.size()
head_dim = embed_dim // heads
B, T_q, C, head_dim

(2, 9, 6, 3)

**Self attention** 

For our encoder, the linear attention is self attention, meaning it allows for each token (gene) to build a relationship with the other tokens. Typically this can be extremely memory expensive as we'd make a TxT matrix in memory. As you'll see, with linear attention we do not need to do that. You might now ask: why even include this variable `kv_input`. This is because we built our linear attention to be both self attention and cross attention. Cross attention is when you're building a relationship between two different inputs. 

In [43]:
kv_input = x_norm
kv_input

tensor([[[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000]],

        [[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
         [-1.0000, -1.0

In [44]:
T_kv = kv_input.size(1)
T_kv

9

**Query** 

Let's start with our query. The query is the current position's "search request." Every layer and head issues queries that can look for relationships. Our query first calculates a linear weight $Q=x_{norm}W^\top+b$, resulting in a vector of $[B,T,C]$. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T,C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T,C_{Heads}]$

Finally we'll apply the ELU+1 function. In linear attention, since we're skipping softmax, we need to ensure that the attention weights stay non-negative like they would with softmax. ELU+1 approximates softmax attention's behavior while keeping the linear complexity benefit.

In [45]:
q_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(q_proj.weight, -0.1)
nn.init.constant_(q_proj.bias, 0)
q_proj.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [46]:
q = q_proj(x_norm).view(B, T_q, heads, head_dim).transpose(1, 2)
q.shape, q

(torch.Size([2, 2, 9, 3]),
 tensor([[[[ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000]],
 
          [[ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000]]],
 
 
         [[[ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [ 0.6000,  0.6000,  0.6000],
           [-0.6000, -0.6000, -0.6000],
           [-0.6000, -0.6000, -0.6000],
    

In [47]:
q = F.elu(q) + 1.0
q.shape, q

(torch.Size([2, 2, 9, 3]),
 tensor([[[[1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000]],
 
          [[1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000]]],
 
 
         [[[1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
           [0.5488, 0.5488, 0.5488],
           [1.6000, 1.6000, 1.6000],
           [0.5488, 0.5488, 0.5488],
  

**Key** 

Next, we calculate the key. The key acts as a matching tag/address for each allowed token. It is compared with the query to produce relevance scores. If this was cross-attention and kv was provided, K would be based on it. Since it's self attention, we again project the normalized X allowing the model to build the second half of the cross. 

We again calculate a linear weight $K=x_{kv\_input}\cdot W^\top+b$, resulting in a vector of $[B,T,C]$. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T,C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T,C_{Heads}]$

Finally we'll apply the ELU+1 function. In linear attention, since we're skipping softmax, we need to ensure that the attention weights stay non-negative like they would with softmax. ELU+1 approximates softmax attention's behavior while keeping the linear complexity benefit.

In [48]:
k_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(k_proj.weight, 0.2)
nn.init.constant_(k_proj.bias, 0)
k_proj.weight

Parameter containing:
tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000]], requires_grad=True)

In [49]:
k = k_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
k.shape, k

(torch.Size([2, 2, 9, 3]),
 tensor([[[[-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000]],
 
          [[-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000]]],
 
 
         [[[-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [-1.2000, -1.2000, -1.2000],
           [-1.2000, -1.2000, -1.2000],
           [ 1.2000,  1.2000,  1.2000],
           [ 1.2000,  1.2000,  1.2000],
    

In [50]:
k = F.elu(k) + 1.0
k

tensor([[[[0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012]],

         [[0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012]]],


        [[[0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000],
          [0.3012, 0.3012, 0.3012],
          [2.2000, 2.2000, 2.2000],
          [2.2000, 2.2000, 2.2000]],

         [[0.3012, 0

**Value** 

Next, we calculate the value. The value is the payload you actually mix in once something matches. It's a learned projection of the token's representation so the model can copy the right kind of information. Similarly, since this is self attention, V will be based on X. 

We again calculate a linear weight $V=x_{kv\_input}\cdot W^\top+b$, resulting in a vector of $[B,T,C]$. We then need to split this across our heads, by splitting $C$, the embedding dimension, across our heads $H$, and then making sure that we have our $[T,C]$ by head by transposing our middle dimensions, resulting in a tensor of $[B,Heads,T,C_{Heads}]$

For V we do not use ELU+1 since the QK will act on V. 

In [51]:
v_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(v_proj.weight, 1.0)
nn.init.constant_(v_proj.bias, 0)
v_proj.weight

Parameter containing:
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]], requires_grad=True)

In [52]:
v = v_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
v.shape, v

(torch.Size([2, 2, 9, 3]),
 tensor([[[[-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000]],
 
          [[-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000]]],
 
 
         [[[-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [-6.0000, -6.0000, -6.0000],
           [-6.0000, -6.0000, -6.0000],
           [ 6.0000,  6.0000,  6.0000],
           [ 6.0000,  6.0000,  6.0000],
    

**Normalization denominator** 

In attention, when using softmax, attention values become probabilities and sum to 1. With linear attention, we need to manually do this normalization.

In softmax attention, the softmax inherently normalizes so weights sum to 1. Linear attention doesn't have that, so we need to create the denominator $z$. We'll do this by summing the tokens per embedding dimension resulting in a $[B,Heads,C_{Heads},1]$ dimension that we can then use in the denominator of our attention calculation

In [53]:
k_sum = k.sum(dim=-2).unsqueeze(-1)
k_sum.shape, k_sum

(torch.Size([2, 2, 3, 1]),
 tensor([[[[10.3060],
           [10.3060],
           [10.3060]],
 
          [[10.3060],
           [10.3060],
           [10.3060]]],
 
 
         [[[12.2048],
           [12.2048],
           [12.2048]],
 
          [[12.2048],
           [12.2048],
           [12.2048]]]], grad_fn=<UnsqueezeBackward0>))

**Denominator** 

We're now ready to calculate the rest of the denominator. The denominator is the sum of attention weights for query across all keys. Each query gets its own normalizing constant, so queries attending to high-magnitude keys don't get inflated outputs. We add a final epsilon to ensure the denominator is not zero. The result from this is that we sum across the head dimensions creating a $[B,Heads,T,1]$ result

In [54]:
z = 1.0 / (q @ k_sum + 1e-6)
z.shape, z

(torch.Size([2, 2, 9, 1]),
 tensor([[[[0.0202],
           [0.0202],
           [0.0589],
           [0.0589],
           [0.0202],
           [0.0202],
           [0.0589],
           [0.0589],
           [0.0202]],
 
          [[0.0202],
           [0.0202],
           [0.0589],
           [0.0589],
           [0.0202],
           [0.0202],
           [0.0589],
           [0.0589],
           [0.0202]]],
 
 
         [[[0.0171],
           [0.0498],
           [0.0171],
           [0.0171],
           [0.0498],
           [0.0498],
           [0.0171],
           [0.0498],
           [0.0498]],
 
          [[0.0171],
           [0.0498],
           [0.0171],
           [0.0171],
           [0.0498],
           [0.0498],
           [0.0171],
           [0.0498],
           [0.0498]]]], grad_fn=<MulBackward0>))

**Numerator**

Now we're ready to complete our numerator. This is just a matter of multiplying Q, K, and V. We'll need to transpose K to get the interaction between the query and key to then multiply against the value. 

Since we're looking to save memory, we actually first multiply the key and value to create head dimension matrixes, and then multiply by the query. This order of operations is part of what saves memory. 

In [55]:
kv = k.transpose(-2, -1) @ v
kv.shape, kv

(torch.Size([2, 2, 3, 3]),
 tensor([[[[43.7642, 43.7642, 43.7642],
           [43.7642, 43.7642, 43.7642],
           [43.7642, 43.7642, 43.7642]],
 
          [[43.7642, 43.7642, 43.7642],
           [43.7642, 43.7642, 43.7642],
           [43.7642, 43.7642, 43.7642]]],
 
 
         [[[58.7713, 58.7713, 58.7713],
           [58.7713, 58.7713, 58.7713],
           [58.7713, 58.7713, 58.7713]],
 
          [[58.7713, 58.7713, 58.7713],
           [58.7713, 58.7713, 58.7713],
           [58.7713, 58.7713, 58.7713]]]], grad_fn=<UnsafeViewBackward0>))

In [56]:
qkv = q @ kv
qkv.shape, qkv

(torch.Size([2, 2, 9, 3]),
 tensor([[[[210.0680, 210.0680, 210.0680],
           [210.0680, 210.0680, 210.0680],
           [ 72.0548,  72.0548,  72.0548],
           [ 72.0549,  72.0549,  72.0549],
           [210.0680, 210.0680, 210.0680],
           [210.0680, 210.0680, 210.0680],
           [ 72.0548,  72.0548,  72.0548],
           [ 72.0548,  72.0548,  72.0548],
           [210.0680, 210.0680, 210.0680]],
 
          [[210.0680, 210.0680, 210.0680],
           [210.0680, 210.0680, 210.0680],
           [ 72.0548,  72.0548,  72.0548],
           [ 72.0549,  72.0549,  72.0549],
           [210.0680, 210.0680, 210.0680],
           [210.0680, 210.0680, 210.0680],
           [ 72.0548,  72.0548,  72.0548],
           [ 72.0548,  72.0548,  72.0548],
           [210.0680, 210.0680, 210.0680]]],
 
 
         [[[282.1023, 282.1023, 282.1023],
           [ 96.7632,  96.7632,  96.7632],
           [282.1022, 282.1022, 282.1022],
           [282.1023, 282.1023, 282.1023],
           [ 96.76

**Linear Attention** 

Now we're ready to normalize our numerator by the denominator. You'll see that because of our extremely consistent values and initialization we're ending up with very consistent values by example even across heads. 

In [57]:
y = qkv * z
y.shape, y

(torch.Size([2, 2, 9, 3]),
 tensor([[[[4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465]],
 
          [[4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465],
           [4.2465, 4.2465, 4.2465]]],
 
 
         [[[4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
           [4.8154, 4.8154, 4.8154],
  

**Collapse heads** 

We now can bring our heads back together. We have to undo our head splitting. First we flip our heads and tokens (genes) so that we have a $[B,T,Heads,C_{Heads}]$ and then we collapse the head and head embeddings to end up with $[B,T,C]$

In [58]:
y = y.transpose(1, 2).contiguous().view(B, T_q, C)
y.shape, y

(torch.Size([2, 9, 6]),
 tensor([[[4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465],
          [4.2465, 4.2465, 4.2465, 4.2465, 4.2465, 4.2465]],
 
         [[4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.8154, 4.8154, 4.8154, 4.8154, 4.8154, 4.8154],
          [4.

**Gating** 

Now we're ready to add sigmoid based gating on top of our attention. This gating uses the Hadamard product of the gate and the attention, which lets the model learn to suppress or pass through attention output per dimension. This allows the model to decide how much of the attention result to actually use for each dimension. We use a learned linear weight and sigmoid to pull gating values between 0 and 1, allowing the model to turn down specific values. 

In [59]:
gate = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.arange(vs).unsqueeze(1)
cols = torch.full((d,), 1.0).unsqueeze(0)
pattern = 0.1*(rows + cols)  

gate.weight = nn.Parameter(pattern)
gate.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000]], requires_grad=True)

In [60]:
y = torch.sigmoid(gate(x_norm)) * y
y.shape, y

(torch.Size([2, 9, 6]),
 tensor([[[1.6656, 1.0135, 0.4362, 0.4505, 0.1963, 0.1514],
          [1.6656, 1.0135, 0.4362, 0.4505, 0.1963, 0.1514],
          [2.8952, 3.2934, 3.4283, 3.9711, 4.0399, 4.1624],
          [2.8952, 3.2934, 3.4283, 3.9711, 4.0399, 4.1624],
          [1.6656, 1.0135, 0.4362, 0.4505, 0.1963, 0.1514],
          [1.6656, 1.0135, 0.4362, 0.4505, 0.1963, 0.1514],
          [2.8952, 3.2934, 3.4283, 3.9711, 4.0399, 4.1624],
          [2.8952, 3.2934, 3.4283, 3.9711, 4.0399, 4.1624],
          [1.6656, 1.0135, 0.4362, 0.4505, 0.1963, 0.1514]],
 
         [[1.8888, 1.1493, 0.4947, 0.5108, 0.2226, 0.1717],
          [3.2832, 3.7347, 3.8876, 4.5031, 4.5811, 4.7201],
          [1.8888, 1.1493, 0.4947, 0.5108, 0.2226, 0.1717],
          [1.8888, 1.1493, 0.4947, 0.5108, 0.2226, 0.1717],
          [3.2832, 3.7347, 3.8876, 4.5031, 4.5811, 4.7201],
          [3.2832, 3.7347, 3.8876, 4.5031, 4.5811, 4.7201],
          [1.8888, 1.1493, 0.4947, 0.5108, 0.2226, 0.1717],
          [3.

**Cross-head final projection** 

Finally, we will now project the gated attention matrix on another final linear layer. This allows the model to learn how to combine information across the different heads. 

In [61]:
c_proj = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1*(rows + 0.01*cols)  

c_proj.weight = nn.Parameter(pattern)
c_proj.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.1100, 0.1100, 0.1100, 0.1100, 0.1100, 0.1100],
        [0.1200, 0.1200, 0.1200, 0.1200, 0.1200, 0.1200],
        [0.1300, 0.1300, 0.1300, 0.1300, 0.1300, 0.1300],
        [0.1400, 0.1400, 0.1400, 0.1400, 0.1400, 0.1400],
        [0.1500, 0.1500, 0.1500, 0.1500, 0.1500, 0.1500]], requires_grad=True)

In [62]:
x_attn = c_proj(y)
x_attn.shape, x_attn

(torch.Size([2, 9, 6]),
 tensor([[[0.4214, 0.4117, 0.6743, 0.8992, 0.3045, 0.2333],
          [0.4214, 0.4117, 0.6743, 0.8992, 0.3045, 0.2333],
          [2.2091, 2.3782, 2.8195, 3.2232, 2.8072, 2.9148],
          [2.2091, 2.3782, 2.8195, 3.2232, 2.8072, 2.9148],
          [0.4214, 0.4117, 0.6743, 0.8992, 0.3045, 0.2333],
          [0.4214, 0.4117, 0.6743, 0.8992, 0.3045, 0.2333],
          [2.2091, 2.3782, 2.8195, 3.2232, 2.8072, 2.9148],
          [2.2091, 2.3782, 2.8195, 3.2232, 2.8072, 2.9148],
          [0.4214, 0.4117, 0.6743, 0.8992, 0.3045, 0.2333]],
 
         [[0.4738, 0.4694, 0.7372, 0.9674, 0.3779, 0.3119],
          [2.5010, 2.6993, 3.1699, 3.6027, 3.2160, 3.3527],
          [0.4738, 0.4694, 0.7372, 0.9674, 0.3779, 0.3119],
          [0.4738, 0.4694, 0.7372, 0.9674, 0.3779, 0.3119],
          [2.5010, 2.6993, 3.1699, 3.6027, 3.2160, 3.3527],
          [2.5010, 2.6993, 3.1699, 3.6027, 3.2160, 3.3527],
          [0.4738, 0.4694, 0.7372, 0.9674, 0.3779, 0.3119],
          [2.

#### Residual Connection

We now have our gated linear attention calculated and will use a residual connection to allow gradients to bypass the attention matrix. With this you'll see how much larger the residual connection impact is on our output compared to our attention. 

In [63]:
x = x + x_attn
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-10.1786, -10.1883,  -9.9257,  -9.7008, -10.2955, -10.3667],
          [ -4.1786,  -4.1883,  -3.9257,  -3.7008,  -4.2955,  -4.3667],
          [  8.8554,   9.0244,   9.4658,   9.8695,   9.4535,   9.5611],
          [  3.6654,   3.8345,   4.2758,   4.6795,   4.2636,   4.3711],
          [-10.1786, -10.1883,  -9.9257,  -9.7008, -10.2955, -10.3667],
          [-10.1786, -10.1883,  -9.9257,  -9.7008, -10.2955, -10.3667],
          [ 12.5435,  12.7126,  13.1539,  13.5576,  13.1416,  13.2492],
          [  7.0013,   7.1704,   7.6117,   8.0154,   7.5995,   7.7070],
          [ -4.1786,  -4.1883,  -3.9257,  -3.7008,  -4.2955,  -4.3667]],
 
         [[-10.1262, -10.1306,  -9.8628,  -9.6326, -10.2221, -10.2881],
          [  3.1282,   3.3265,   3.7970,   4.2299,   3.8432,   3.9799],
          [ -4.1262,  -4.1306,  -3.8628,  -3.6326,  -4.2221,  -4.2881],
          [-10.1262, -10.1306,  -9.8628,  -9.6326, -10.2221, -10.2881],
          [  5.4205,   5.6187,   6.08

#### RMSNorm 2
Now that we've calculated the attention, we'll do another round of normalization. We'll use RMSNorm again. 

In [64]:
rms2= RMSNorm(embed_dim)
rms2.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [65]:
x_norm2 = rms2(x)
x_norm2.shape, x_norm2

(torch.Size([2, 9, 6]),
 tensor([[[-1.0066, -1.0076, -0.9816, -0.9593, -1.0182, -1.0252],
          [-1.0153, -1.0177, -0.9539, -0.8992, -1.0437, -1.0610],
          [ 0.9443,  0.9623,  1.0094,  1.0524,  1.0081,  1.0196],
          [ 0.8737,  0.9140,  1.0192,  1.1154,  1.0163,  1.0419],
          [-1.0066, -1.0076, -0.9816, -0.9593, -1.0182, -1.0252],
          [-1.0066, -1.0076, -0.9816, -0.9593, -1.0182, -1.0252],
          [ 0.9601,  0.9731,  1.0069,  1.0378,  1.0059,  1.0142],
          [ 0.9304,  0.9529,  1.0115,  1.0651,  1.0099,  1.0242],
          [-1.0153, -1.0177, -0.9539, -0.8992, -1.0437, -1.0610]],
 
         [[-1.0080, -1.0084, -0.9817, -0.9588, -1.0175, -1.0241],
          [ 0.8372,  0.8903,  1.0162,  1.1320,  1.0285,  1.0651],
          [-1.0188, -1.0199, -0.9538, -0.8969, -1.0425, -1.0588],
          [-1.0080, -1.0084, -0.9817, -0.9588, -1.0175, -1.0241],
          [ 0.9002,  0.9331,  1.0113,  1.0831,  1.0189,  1.0416],
          [ 0.8882,  0.9250,  1.0123,  1.0926,  1

#### SwiGLU 

Next, we'll introduce our scaling nonlinearity layers. Traditionally this was a MLP, but we replaced it with a swish-gated linear unit, or SwiGLU. SwiGLU replaces the single linear transform in a standard MLP with a gated pathway with a result as follows:

$$\begin{aligned}
\text{SiLU}(Z) &= Z \odot \sigma(Z) \\
\text{gate} &= \text{SiLU}(x W_g^\top) \\
H &= x W_u^\top \\
y &= (\text{gate} \odot H) W_d^\top
\end{aligned}$$

The Hadamard product based gating lets the network learn to selectively amplify or suppress features before the final projection, giving it more expressive power per parameter than a standard two-layer MLP with ReLU/GELU at the cost of some extra compute.

*You'll notice that SwiGLU has 3 weight matrices $W_g, W_u, W_d$, instead of the typical 2 we use in MLP. In production code, you might see helper functions that convert MLP ratios to SwiGLU ratios using 2/3 multiples to maintain the number of parameters*

In [66]:
hidden_dim = 8

**Gate** 

We'll start by initializing our gating weight $W_g$ and calculating our gate. The gate will need to scale up to our hidden dimension as it will multiply directly against our weighted input. 

In [67]:
wg = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wg.weight, 0.5)
wg.weight

Parameter containing:
tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000]], requires_grad=True)

In [68]:
xwg = wg(x_norm2)
xwg.shape, xwg

(torch.Size([2, 9, 8]),
 tensor([[[-2.9992, -2.9992, -2.9992, -2.9992, -2.9992, -2.9992, -2.9992,
           -2.9992],
          [-2.9954, -2.9954, -2.9954, -2.9954, -2.9954, -2.9954, -2.9954,
           -2.9954],
          [ 2.9980,  2.9980,  2.9980,  2.9980,  2.9980,  2.9980,  2.9980,
            2.9980],
          [ 2.9902,  2.9902,  2.9902,  2.9902,  2.9902,  2.9902,  2.9902,
            2.9902],
          [-2.9992, -2.9992, -2.9992, -2.9992, -2.9992, -2.9992, -2.9992,
           -2.9992],
          [-2.9992, -2.9992, -2.9992, -2.9992, -2.9992, -2.9992, -2.9992,
           -2.9992],
          [ 2.9990,  2.9990,  2.9990,  2.9990,  2.9990,  2.9990,  2.9990,
            2.9990],
          [ 2.9970,  2.9970,  2.9970,  2.9970,  2.9970,  2.9970,  2.9970,
            2.9970],
          [-2.9954, -2.9954, -2.9954, -2.9954, -2.9954, -2.9954, -2.9954,
           -2.9954]],
 
         [[-2.9992, -2.9992, -2.9992, -2.9992, -2.9992, -2.9992, -2.9992,
           -2.9992],
          [ 2.9847,  2.

**SiLU Non-Linearity**

SiLU will pull our negative values closer to zero. Values above 1 will remain almost linear. When combined with the learned weights, you can quickly see how this becomes a gate. 

In [69]:
xwg = F.silu(xwg)
xwg.shape, xwg

(torch.Size([2, 9, 8]),
 tensor([[[-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [-0.1427, -0.1427, -0.1427, -0.1427, -0.1427, -0.1427, -0.1427,
           -0.1427],
          [ 2.8556,  2.8556,  2.8556,  2.8556,  2.8556,  2.8556,  2.8556,
            2.8556],
          [ 2.8471,  2.8471,  2.8471,  2.8471,  2.8471,  2.8471,  2.8471,
            2.8471],
          [-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [ 2.8566,  2.8566,  2.8566,  2.8566,  2.8566,  2.8566,  2.8566,
            2.8566],
          [ 2.8544,  2.8544,  2.8544,  2.8544,  2.8544,  2.8544,  2.8544,
            2.8544],
          [-0.1427, -0.1427, -0.1427, -0.1427, -0.1427, -0.1427, -0.1427,
           -0.1427]],
 
         [[-0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423, -0.1423,
           -0.1423],
          [ 2.8410,  2.

**Weighted Input** 

Now we'll need to scale up our input to the hidden dimension. We'll use a weighted layer $W_u$ allowing the model to determine how to use the different channels to create the new dimensions

In [70]:
wu = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wu.weight, -0.1)
wu.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [71]:
xwu = wu(x_norm2)
xwu.shape, xwu

(torch.Size([2, 9, 8]),
 tensor([[[ 0.5998,  0.5998,  0.5998,  0.5998,  0.5998,  0.5998,  0.5998,
            0.5998],
          [ 0.5991,  0.5991,  0.5991,  0.5991,  0.5991,  0.5991,  0.5991,
            0.5991],
          [-0.5996, -0.5996, -0.5996, -0.5996, -0.5996, -0.5996, -0.5996,
           -0.5996],
          [-0.5980, -0.5980, -0.5980, -0.5980, -0.5980, -0.5980, -0.5980,
           -0.5980],
          [ 0.5998,  0.5998,  0.5998,  0.5998,  0.5998,  0.5998,  0.5998,
            0.5998],
          [ 0.5998,  0.5998,  0.5998,  0.5998,  0.5998,  0.5998,  0.5998,
            0.5998],
          [-0.5998, -0.5998, -0.5998, -0.5998, -0.5998, -0.5998, -0.5998,
           -0.5998],
          [-0.5994, -0.5994, -0.5994, -0.5994, -0.5994, -0.5994, -0.5994,
           -0.5994],
          [ 0.5991,  0.5991,  0.5991,  0.5991,  0.5991,  0.5991,  0.5991,
            0.5991]],
 
         [[ 0.5998,  0.5998,  0.5998,  0.5998,  0.5998,  0.5998,  0.5998,
            0.5998],
          [-0.5969, -0.

**Apply Gate** Now we'll go ahead and apply the gate. We take the Hadamard product which allows the model to gate each value of the scaled up projection. 

In [72]:
xw = xwg * xwu
xw.shape, xw

(torch.Size([2, 9, 8]),
 tensor([[[-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-0.0855, -0.0855, -0.0855, -0.0855, -0.0855, -0.0855, -0.0855,
           -0.0855],
          [-1.7122, -1.7122, -1.7122, -1.7122, -1.7122, -1.7122, -1.7122,
           -1.7122],
          [-1.7027, -1.7027, -1.7027, -1.7027, -1.7027, -1.7027, -1.7027,
           -1.7027],
          [-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-1.7134, -1.7134, -1.7134, -1.7134, -1.7134, -1.7134, -1.7134,
           -1.7134],
          [-1.7109, -1.7109, -1.7109, -1.7109, -1.7109, -1.7109, -1.7109,
           -1.7109],
          [-0.0855, -0.0855, -0.0855, -0.0855, -0.0855, -0.0855, -0.0855,
           -0.0855]],
 
         [[-0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854, -0.0854,
           -0.0854],
          [-1.6959, -1.

**Project Down** 

Now we need to project back down to our embedding dimension. We'll use a final weighted $W_d$ layer to determine how to project back down. This is similar to the final layer of an MLP. 

In [73]:
wd = nn.Linear(hidden_dim, embed_dim, bias=False)
nn.init.constant_(wd.weight, 0.33)
wd.weight

Parameter containing:
tensor([[0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300]],
       requires_grad=True)

In [74]:
xswig = wd(xw)
xswig.shape, xswig

(torch.Size([2, 9, 6]),
 tensor([[[-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-0.2257, -0.2257, -0.2257, -0.2257, -0.2257, -0.2257],
          [-4.5203, -4.5203, -4.5203, -4.5203, -4.5203, -4.5203],
          [-4.4951, -4.4951, -4.4951, -4.4951, -4.4951, -4.4951],
          [-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-4.5234, -4.5234, -4.5234, -4.5234, -4.5234, -4.5234],
          [-4.5168, -4.5168, -4.5168, -4.5168, -4.5168, -4.5168],
          [-0.2257, -0.2257, -0.2257, -0.2257, -0.2257, -0.2257]],
 
         [[-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-4.4772, -4.4772, -4.4772, -4.4772, -4.4772, -4.4772],
          [-0.2257, -0.2257, -0.2257, -0.2257, -0.2257, -0.2257],
          [-0.2254, -0.2254, -0.2254, -0.2254, -0.2254, -0.2254],
          [-4.5076, -4.5076, -4.5076, -4.5076, -4.5076, -4.5076],
          [-4.5029, -4.5029, -4.5029, -4.5029, -4

#### Residual Connection 2

We now have our SwiGLU-based projection calculated and will use a residual connection to allow gradients to bypass the SwiGLU calculations. With this you'll see how much larger the residual connection impact is on our output compared to our SwiGLU output. 

In [75]:
x = x + xswig
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-10.4040, -10.4137, -10.1511,  -9.9262, -10.5209, -10.5921],
          [ -4.4043,  -4.4139,  -4.1514,  -3.9265,  -4.5212,  -4.5924],
          [  4.3350,   4.5041,   4.9454,   5.3491,   4.9332,   5.0407],
          [ -0.8297,  -0.6606,  -0.2193,   0.1844,  -0.2316,  -0.1240],
          [-10.4040, -10.4137, -10.1511,  -9.9262, -10.5209, -10.5921],
          [-10.4040, -10.4137, -10.1511,  -9.9262, -10.5209, -10.5921],
          [  8.0201,   8.1892,   8.6305,   9.0342,   8.6182,   8.7258],
          [  2.4845,   2.6536,   3.0949,   3.4986,   3.0826,   3.1902],
          [ -4.4043,  -4.4139,  -4.1514,  -3.9265,  -4.5212,  -4.5924]],
 
         [[-10.3516, -10.3560, -10.0882,  -9.8580, -10.4475, -10.5135],
          [ -1.3490,  -1.1507,  -0.6802,  -0.2473,  -0.6341,  -0.4973],
          [ -4.3518,  -4.3563,  -4.0885,  -3.8583,  -4.4478,  -4.5137],
          [-10.3516, -10.3560, -10.0882,  -9.8580, -10.4475, -10.5135],
          [  0.9129,   1.1111,   1.58

### Final Layer Normalization
The previous layers can run sequentially for as many layers as is configured. The more layers, the "deeper" the network becomes. Once all the layers have completed, we're ready for a final normalization. Like previous normalizations this will pull our values together to focus on the variance. This output then becomes the latent representation of the context or target. 

In [76]:
rmsf = RMSNorm(embed_dim)
rmsf.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [77]:
x = rmsf(x)
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [-1.0146, -1.0168, -0.9563, -0.9045, -1.0415, -1.0579],
          [ 0.8914,  0.9262,  1.0169,  1.1000,  1.0144,  1.0365],
          [-1.7992, -1.4326, -0.4755,  0.3999, -0.5021, -0.2689],
          [-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [ 0.9388,  0.9586,  1.0102,  1.0575,  1.0088,  1.0214],
          [ 0.8228,  0.8787,  1.0249,  1.1586,  1.0208,  1.0564],
          [-1.0146, -1.0168, -0.9563, -0.9045, -1.0415, -1.0579]],
 
         [[-1.0078, -1.0082, -0.9821, -0.9597, -1.0171, -1.0235],
          [-1.5903, -1.3566, -0.8019, -0.2915, -0.7475, -0.5863],
          [-1.0179, -1.0189, -0.9563, -0.9024, -1.0403, -1.0557],
          [-1.0078, -1.0082, -0.9821, -0.9597, -1.0171, -1.0235],
          [ 0.5894,  0.7175,  1.0213,  1.3008,  1.0510,  1.1393],
          [ 0.2983,  0.5068,  1.0016,  1.4569,  1

### Output Latent
Now we've fully processed our input to create the context latent.  This can now either be used for downstream models, or passed into the masked predictor during our encoder training.

In [78]:
context_latent = x
context_latent.shape, context_latent

(torch.Size([2, 9, 6]),
 tensor([[[-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [-1.0146, -1.0168, -0.9563, -0.9045, -1.0415, -1.0579],
          [ 0.8914,  0.9262,  1.0169,  1.1000,  1.0144,  1.0365],
          [-1.7992, -1.4326, -0.4755,  0.3999, -0.5021, -0.2689],
          [-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [ 0.9388,  0.9586,  1.0102,  1.0575,  1.0088,  1.0214],
          [ 0.8228,  0.8787,  1.0249,  1.1586,  1.0208,  1.0564],
          [-1.0146, -1.0168, -0.9563, -0.9045, -1.0415, -1.0579]],
 
         [[-1.0078, -1.0082, -0.9821, -0.9597, -1.0171, -1.0235],
          [-1.5903, -1.3566, -0.8019, -0.2915, -0.7475, -0.5863],
          [-1.0179, -1.0189, -0.9563, -0.9024, -1.0403, -1.0557],
          [-1.0078, -1.0082, -0.9821, -0.9597, -1.0171, -1.0235],
          [ 0.5894,  0.7175,  1.0213,  1.3008,  1.0510,  1.1393],
          [ 0.2983,  0.5068,  1.0016,  1.4569,  1

## Masked Predictor
During training, when we have a masked input to the encoder, we process the output with a masked predictor. The masked predictor is a lightweight head that takes the context encoder's full output and refines it into predictions specifically at the masked positions, keeping the reconstruction objective separate from the encoder's learned representations so the student doesn't overfit to predicting missing tokens at the expense of learning generally useful gene embeddings.

This predictor is essentially the same transformer block, with just a final linear head added. Since the layers are identical in architecture and computation, we'll only call out any specific differences.  

### Transformer Block 
The masked predictor uses the same transformer block consisting of 
1. RMS normalization
2. Gated linear self-attention
3. RMS normalization
4. SwiGLU

Residual connections provide gradient bypassing around the attention and SwiGLU layers. This set of 4 units is repeated based on how many layers are configured. 

#### RMSNorm M1

In [79]:
rms_m1 = RMSNorm(embed_dim)

rms_m1.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [80]:
x_norm = rms_m1(x)
x_norm.shape, x_norm

(torch.Size([2, 9, 6]),
 tensor([[[-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [-1.0146, -1.0168, -0.9563, -0.9045, -1.0415, -1.0579],
          [ 0.8914,  0.9262,  1.0169,  1.1000,  1.0144,  1.0365],
          [-1.7992, -1.4326, -0.4755,  0.3999, -0.5021, -0.2689],
          [-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [ 0.9388,  0.9586,  1.0102,  1.0575,  1.0088,  1.0214],
          [ 0.8228,  0.8787,  1.0249,  1.1586,  1.0208,  1.0564],
          [-1.0146, -1.0168, -0.9563, -0.9045, -1.0415, -1.0579]],
 
         [[-1.0078, -1.0082, -0.9821, -0.9597, -1.0171, -1.0235],
          [-1.5903, -1.3566, -0.8019, -0.2915, -0.7475, -0.5863],
          [-1.0179, -1.0189, -0.9563, -0.9024, -1.0403, -1.0557],
          [-1.0078, -1.0082, -0.9821, -0.9597, -1.0171, -1.0235],
          [ 0.5894,  0.7175,  1.0213,  1.3008,  1.0510,  1.1393],
          [ 0.2983,  0.5068,  1.0016,  1.4569,  1

#### Multi-Headed Gated Linear Attention

In [81]:
B, T_q, C = x_norm.size()
head_dim = embed_dim // heads
B, T_q, C, head_dim

(2, 9, 6, 3)

**Self-attention**

In [82]:
kv_input = x_norm
T_kv = kv_input.size(1)
kv_input, T_kv

(tensor([[[-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [-1.0146, -1.0168, -0.9563, -0.9045, -1.0415, -1.0579],
          [ 0.8914,  0.9262,  1.0169,  1.1000,  1.0144,  1.0365],
          [-1.7992, -1.4326, -0.4755,  0.3999, -0.5021, -0.2689],
          [-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
          [ 0.9388,  0.9586,  1.0102,  1.0575,  1.0088,  1.0214],
          [ 0.8228,  0.8787,  1.0249,  1.1586,  1.0208,  1.0564],
          [-1.0146, -1.0168, -0.9563, -0.9045, -1.0415, -1.0579]],
 
         [[-1.0078, -1.0082, -0.9821, -0.9597, -1.0171, -1.0235],
          [-1.5903, -1.3566, -0.8019, -0.2915, -0.7475, -0.5863],
          [-1.0179, -1.0189, -0.9563, -0.9024, -1.0403, -1.0557],
          [-1.0078, -1.0082, -0.9821, -0.9597, -1.0171, -1.0235],
          [ 0.5894,  0.7175,  1.0213,  1.3008,  1.0510,  1.1393],
          [ 0.2983,  0.5068,  1.0016,  1.4569,  1.0501,  1.1940],
       

**Query**

In [83]:
q_proj_m = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(q_proj_m.weight, -0.1)
nn.init.constant_(q_proj_m.bias, 0)
q_proj_m.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [84]:
q = q_proj_m(x_norm).view(B, T_q, heads, head_dim).transpose(1, 2)
q.shape, q

(torch.Size([2, 2, 9, 3]),
 tensor([[[[ 0.5999,  0.5999,  0.5999],
           [ 0.5992,  0.5992,  0.5992],
           [-0.5985, -0.5985, -0.5985],
           [ 0.4079,  0.4079,  0.4079],
           [ 0.5999,  0.5999,  0.5999],
           [ 0.5999,  0.5999,  0.5999],
           [-0.5995, -0.5995, -0.5995],
           [-0.5962, -0.5962, -0.5962],
           [ 0.5992,  0.5992,  0.5992]],
 
          [[ 0.5999,  0.5999,  0.5999],
           [ 0.5992,  0.5992,  0.5992],
           [-0.5985, -0.5985, -0.5985],
           [ 0.4079,  0.4079,  0.4079],
           [ 0.5999,  0.5999,  0.5999],
           [ 0.5999,  0.5999,  0.5999],
           [-0.5995, -0.5995, -0.5995],
           [-0.5962, -0.5962, -0.5962],
           [ 0.5992,  0.5992,  0.5992]]],
 
 
         [[[ 0.5999,  0.5999,  0.5999],
           [ 0.5374,  0.5374,  0.5374],
           [ 0.5992,  0.5992,  0.5992],
           [ 0.5999,  0.5999,  0.5999],
           [-0.5819, -0.5819, -0.5819],
           [-0.5508, -0.5508, -0.5508],
    

In [85]:
q = F.elu(q) + 1.0
q.shape, q

(torch.Size([2, 2, 9, 3]),
 tensor([[[[1.5999, 1.5999, 1.5999],
           [1.5992, 1.5992, 1.5992],
           [0.5496, 0.5496, 0.5496],
           [1.4079, 1.4079, 1.4079],
           [1.5999, 1.5999, 1.5999],
           [1.5999, 1.5999, 1.5999],
           [0.5491, 0.5491, 0.5491],
           [0.5509, 0.5509, 0.5509],
           [1.5992, 1.5992, 1.5992]],
 
          [[1.5999, 1.5999, 1.5999],
           [1.5992, 1.5992, 1.5992],
           [0.5496, 0.5496, 0.5496],
           [1.4079, 1.4079, 1.4079],
           [1.5999, 1.5999, 1.5999],
           [1.5999, 1.5999, 1.5999],
           [0.5491, 0.5491, 0.5491],
           [0.5509, 0.5509, 0.5509],
           [1.5992, 1.5992, 1.5992]]],
 
 
         [[[1.5999, 1.5999, 1.5999],
           [1.5374, 1.5374, 1.5374],
           [1.5992, 1.5992, 1.5992],
           [1.5999, 1.5999, 1.5999],
           [0.5588, 0.5588, 0.5588],
           [0.5765, 0.5765, 0.5765],
           [1.5999, 1.5999, 1.5999],
           [0.5489, 0.5489, 0.5489],
  

**Key**

In [86]:
k_proj_m = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(k_proj_m.weight, 0.2)
nn.init.constant_(k_proj_m.bias, 0)
k_proj_m.weight

Parameter containing:
tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000]], requires_grad=True)

In [87]:
k = k_proj_m(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
k.shape, k

(torch.Size([2, 2, 9, 3]),
 tensor([[[[-1.1997, -1.1997, -1.1997],
           [-1.1983, -1.1983, -1.1983],
           [ 1.1971,  1.1971,  1.1971],
           [-0.8157, -0.8157, -0.8157],
           [-1.1997, -1.1997, -1.1997],
           [-1.1997, -1.1997, -1.1997],
           [ 1.1991,  1.1991,  1.1991],
           [ 1.1924,  1.1924,  1.1924],
           [-1.1983, -1.1983, -1.1983]],
 
          [[-1.1997, -1.1997, -1.1997],
           [-1.1983, -1.1983, -1.1983],
           [ 1.1971,  1.1971,  1.1971],
           [-0.8157, -0.8157, -0.8157],
           [-1.1997, -1.1997, -1.1997],
           [-1.1997, -1.1997, -1.1997],
           [ 1.1991,  1.1991,  1.1991],
           [ 1.1924,  1.1924,  1.1924],
           [-1.1983, -1.1983, -1.1983]]],
 
 
         [[[-1.1997, -1.1997, -1.1997],
           [-1.0748, -1.0748, -1.0748],
           [-1.1983, -1.1983, -1.1983],
           [-1.1997, -1.1997, -1.1997],
           [ 1.1639,  1.1639,  1.1639],
           [ 1.1015,  1.1015,  1.1015],
    

In [88]:
k = F.elu(k) + 1.0
k

tensor([[[[0.3013, 0.3013, 0.3013],
          [0.3017, 0.3017, 0.3017],
          [2.1971, 2.1971, 2.1971],
          [0.4423, 0.4423, 0.4423],
          [0.3013, 0.3013, 0.3013],
          [0.3013, 0.3013, 0.3013],
          [2.1991, 2.1991, 2.1991],
          [2.1924, 2.1924, 2.1924],
          [0.3017, 0.3017, 0.3017]],

         [[0.3013, 0.3013, 0.3013],
          [0.3017, 0.3017, 0.3017],
          [2.1971, 2.1971, 2.1971],
          [0.4423, 0.4423, 0.4423],
          [0.3013, 0.3013, 0.3013],
          [0.3013, 0.3013, 0.3013],
          [2.1991, 2.1991, 2.1991],
          [2.1924, 2.1924, 2.1924],
          [0.3017, 0.3017, 0.3017]]],


        [[[0.3013, 0.3013, 0.3013],
          [0.3414, 0.3414, 0.3414],
          [0.3017, 0.3017, 0.3017],
          [0.3013, 0.3013, 0.3013],
          [2.1639, 2.1639, 2.1639],
          [2.1015, 2.1015, 2.1015],
          [0.3013, 0.3013, 0.3013],
          [2.1997, 2.1997, 2.1997],
          [2.1993, 2.1993, 2.1993]],

         [[0.3013, 0

**Value**

In [89]:
v_proj_m = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(v_proj_m.weight, 1.0)
nn.init.constant_(v_proj_m.bias, 0)
v_proj_m.weight

Parameter containing:
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]], requires_grad=True)

In [90]:
v = v_proj_m(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
v.shape, v

(torch.Size([2, 2, 9, 3]),
 tensor([[[[-5.9985, -5.9985, -5.9985],
           [-5.9917, -5.9917, -5.9917],
           [ 5.9855,  5.9855,  5.9855],
           [-4.0785, -4.0785, -4.0785],
           [-5.9985, -5.9985, -5.9985],
           [-5.9985, -5.9985, -5.9985],
           [ 5.9953,  5.9953,  5.9953],
           [ 5.9622,  5.9622,  5.9622],
           [-5.9917, -5.9917, -5.9917]],
 
          [[-5.9985, -5.9985, -5.9985],
           [-5.9917, -5.9917, -5.9917],
           [ 5.9855,  5.9855,  5.9855],
           [-4.0785, -4.0785, -4.0785],
           [-5.9985, -5.9985, -5.9985],
           [-5.9985, -5.9985, -5.9985],
           [ 5.9953,  5.9953,  5.9953],
           [ 5.9622,  5.9622,  5.9622],
           [-5.9917, -5.9917, -5.9917]]],
 
 
         [[[-5.9985, -5.9985, -5.9985],
           [-5.3741, -5.3741, -5.3741],
           [-5.9916, -5.9916, -5.9916],
           [-5.9985, -5.9985, -5.9985],
           [ 5.8193,  5.8193,  5.8193],
           [ 5.5077,  5.5077,  5.5077],
    

**Normalize Denominator**

In [91]:
k_sum = k.sum(dim=-2).unsqueeze(-1)
k_sum.shape, k_sum

(torch.Size([2, 2, 3, 1]),
 tensor([[[[ 8.5382],
           [ 8.5382],
           [ 8.5382]],
 
          [[ 8.5382],
           [ 8.5382],
           [ 8.5382]]],
 
 
         [[[10.2113],
           [10.2113],
           [10.2113]],
 
          [[10.2113],
           [10.2113],
           [10.2113]]]], grad_fn=<UnsqueezeBackward0>))

**Denominator**

In [92]:
z = 1.0 / (q @ k_sum + 1e-6)
z.shape, z

(torch.Size([2, 2, 9, 1]),
 tensor([[[[0.0244],
           [0.0244],
           [0.0710],
           [0.0277],
           [0.0244],
           [0.0244],
           [0.0711],
           [0.0709],
           [0.0244]],
 
          [[0.0244],
           [0.0244],
           [0.0710],
           [0.0277],
           [0.0244],
           [0.0244],
           [0.0711],
           [0.0709],
           [0.0244]]],
 
 
         [[[0.0204],
           [0.0212],
           [0.0204],
           [0.0204],
           [0.0584],
           [0.0566],
           [0.0204],
           [0.0595],
           [0.0595]],
 
          [[0.0204],
           [0.0212],
           [0.0204],
           [0.0204],
           [0.0584],
           [0.0566],
           [0.0204],
           [0.0595],
           [0.0595]]]], grad_fn=<MulBackward0>))

**Numerator**

In [93]:
kv = k.transpose(-2, -1) @ v
kv.shape, kv

(torch.Size([2, 2, 3, 3]),
 tensor([[[[28.5654, 28.5654, 28.5654],
           [28.5654, 28.5654, 28.5654],
           [28.5654, 28.5654, 28.5654]],
 
          [[28.5654, 28.5654, 28.5654],
           [28.5654, 28.5654, 28.5654],
           [28.5654, 28.5654, 28.5654]]],
 
 
         [[[41.4850, 41.4850, 41.4850],
           [41.4850, 41.4850, 41.4850],
           [41.4850, 41.4850, 41.4850]],
 
          [[41.4850, 41.4850, 41.4850],
           [41.4850, 41.4850, 41.4850],
           [41.4850, 41.4850, 41.4850]]]], grad_fn=<UnsafeViewBackward0>))

In [94]:
qkv = q @ kv
qkv.shape, qkv

(torch.Size([2, 2, 9, 3]),
 tensor([[[[137.1015, 137.1015, 137.1015],
           [137.0429, 137.0429, 137.0429],
           [ 47.0995,  47.0995,  47.0995],
           [120.6477, 120.6477, 120.6477],
           [137.1015, 137.1015, 137.1015],
           [137.1015, 137.1015, 137.1015],
           [ 47.0532,  47.0532,  47.0532],
           [ 47.2090,  47.2090,  47.2090],
           [137.0429, 137.0429, 137.0429]],
 
          [[137.1015, 137.1015, 137.1015],
           [137.0429, 137.0429, 137.0429],
           [ 47.0995,  47.0995,  47.0995],
           [120.6477, 120.6477, 120.6477],
           [137.1015, 137.1015, 137.1015],
           [137.1015, 137.1015, 137.1015],
           [ 47.0532,  47.0532,  47.0532],
           [ 47.2090,  47.2090,  47.2090],
           [137.0429, 137.0429, 137.0429]]],
 
 
         [[[199.1099, 199.1099, 199.1099],
           [191.3389, 191.3389, 191.3389],
           [199.0233, 199.0233, 199.0233],
           [199.1099, 199.1099, 199.1099],
           [ 69.54

**Linear Attention**

In [95]:
y = qkv * z
y.shape, y

(torch.Size([2, 2, 9, 3]),
 tensor([[[[3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456]],
 
          [[3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456],
           [3.3456, 3.3456, 3.3456]]],
 
 
         [[[4.0627, 4.0627, 4.0627],
           [4.0627, 4.0627, 4.0627],
           [4.0627, 4.0627, 4.0627],
           [4.0627, 4.0627, 4.0627],
           [4.0627, 4.0627, 4.0627],
           [4.0627, 4.0627, 4.0627],
           [4.0627, 4.0627, 4.0627],
           [4.0627, 4.0627, 4.0627],
  

**Collapse Heads**

In [96]:
y = y.transpose(1, 2).contiguous().view(B, T_q, C)
y.shape, y

(torch.Size([2, 9, 6]),
 tensor([[[3.3456, 3.3456, 3.3456, 3.3456, 3.3456, 3.3456],
          [3.3456, 3.3456, 3.3456, 3.3456, 3.3456, 3.3456],
          [3.3456, 3.3456, 3.3456, 3.3456, 3.3456, 3.3456],
          [3.3456, 3.3456, 3.3456, 3.3456, 3.3456, 3.3456],
          [3.3456, 3.3456, 3.3456, 3.3456, 3.3456, 3.3456],
          [3.3456, 3.3456, 3.3456, 3.3456, 3.3456, 3.3456],
          [3.3456, 3.3456, 3.3456, 3.3456, 3.3456, 3.3456],
          [3.3456, 3.3456, 3.3456, 3.3456, 3.3456, 3.3456],
          [3.3456, 3.3456, 3.3456, 3.3456, 3.3456, 3.3456]],
 
         [[4.0627, 4.0627, 4.0627, 4.0627, 4.0627, 4.0627],
          [4.0627, 4.0627, 4.0627, 4.0627, 4.0627, 4.0627],
          [4.0627, 4.0627, 4.0627, 4.0627, 4.0627, 4.0627],
          [4.0627, 4.0627, 4.0627, 4.0627, 4.0627, 4.0627],
          [4.0627, 4.0627, 4.0627, 4.0627, 4.0627, 4.0627],
          [4.0627, 4.0627, 4.0627, 4.0627, 4.0627, 4.0627],
          [4.0627, 4.0627, 4.0627, 4.0627, 4.0627, 4.0627],
          [4.

**Gating**

In [97]:
gate_m = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.arange(vs).unsqueeze(1)
cols = torch.full((d,), 1.0).unsqueeze(0)
pattern = 0.1*(rows + cols)  

gate_m.weight = nn.Parameter(pattern)
gate_m.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000, 0.4000, 0.4000, 0.4000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.6000, 0.6000, 0.6000, 0.6000, 0.6000, 0.6000]], requires_grad=True)

In [98]:
y = torch.sigmoid(gate_m(x_norm)) * y
y.shape, y

(torch.Size([2, 9, 6]),
 tensor([[[1.3490, 0.7718, 0.3451, 0.3431, 0.1692, 0.1300],
          [1.3495, 0.7726, 0.3458, 0.3439, 0.1697, 0.1305],
          [2.3129, 2.5667, 2.7010, 3.1195, 3.1957, 3.2844],
          [1.5060, 1.0227, 0.5683, 0.6612, 0.4085, 0.3795],
          [1.3490, 0.7718, 0.3451, 0.3431, 0.1692, 0.1300],
          [1.3490, 0.7718, 0.3451, 0.3431, 0.1692, 0.1300],
          [2.3136, 2.5678, 2.7025, 3.1203, 3.1964, 3.2847],
          [2.3112, 2.5639, 2.6973, 3.1175, 3.1940, 3.2835],
          [1.3495, 0.7726, 0.3458, 0.3439, 0.1697, 0.1305]],
 
         [[1.6381, 0.9372, 0.4191, 0.4166, 0.2054, 0.1579],
          [1.6995, 1.0303, 0.4949, 0.5197, 0.2756, 0.2256],
          [1.6388, 0.9382, 0.4199, 0.4177, 0.2061, 0.1585],
          [1.6381, 0.9372, 0.4191, 0.4166, 0.2054, 0.1579],
          [2.7941, 3.0924, 3.2479, 3.7706, 3.8656, 3.9807],
          [2.7668, 3.0457, 3.1853, 3.7350, 3.8343, 3.9642],
          [1.6381, 0.9372, 0.4191, 0.4166, 0.2054, 0.1579],
          [2.

**Cross-head final projection**

In [99]:
c_proj_m = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols  = torch.arange(d).unsqueeze(1)
pattern = 1*(rows + 0.01*cols)  

c_proj_m.weight = nn.Parameter(pattern)
c_proj_m.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
        [0.1100, 0.1100, 0.1100, 0.1100, 0.1100, 0.1100],
        [0.1200, 0.1200, 0.1200, 0.1200, 0.1200, 0.1200],
        [0.1300, 0.1300, 0.1300, 0.1300, 0.1300, 0.1300],
        [0.1400, 0.1400, 0.1400, 0.1400, 0.1400, 0.1400],
        [0.1500, 0.1500, 0.1500, 0.1500, 0.1500, 0.1500]], requires_grad=True)

In [100]:
x_attn = c_proj_m(y)
x_attn.shape, x_attn

(torch.Size([2, 9, 6]),
 tensor([[[0.2351, 0.3805, 0.7506, 0.0096, 0.2137, 0.1638],
          [0.2355, 0.3809, 0.7510, 0.0101, 0.2142, 0.1644],
          [1.6423, 1.9284, 2.4392, 1.8390, 2.1837, 2.2746],
          [0.3789, 0.5386, 0.9231, 0.1966, 0.4150, 0.3795],
          [0.2351, 0.3805, 0.7506, 0.0096, 0.2137, 0.1638],
          [0.2351, 0.3805, 0.7506, 0.0096, 0.2137, 0.1638],
          [1.6429, 1.9290, 2.4398, 1.8397, 2.1845, 2.2754],
          [1.6411, 1.9270, 2.4377, 1.8373, 2.1820, 2.2727],
          [0.2355, 0.3809, 0.7510, 0.0101, 0.2142, 0.1644]],
 
         [[0.3018, 0.4537, 0.8305, 0.0962, 0.3069, 0.2638],
          [0.3489, 0.5056, 0.8871, 0.1575, 0.3729, 0.3345],
          [0.3022, 0.4543, 0.8311, 0.0969, 0.3076, 0.2645],
          [0.3018, 0.4537, 0.8305, 0.0962, 0.3069, 0.2638],
          [1.9995, 2.3212, 2.8677, 2.3032, 2.6837, 2.8103],
          [1.9774, 2.2970, 2.8413, 2.2746, 2.6529, 2.7773],
          [0.3018, 0.4537, 0.8305, 0.0962, 0.3069, 0.2638],
          [2.

#### Residual Connection

In [101]:
x = x + x_attn
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-0.7713, -0.6269, -0.2314, -0.9506, -0.8041, -0.8608],
          [-0.7791, -0.6359, -0.2053, -0.8944, -0.8273, -0.8935],
          [ 2.5338,  2.8546,  3.4561,  2.9389,  3.1981,  3.3112],
          [-1.4203, -0.8939,  0.4476,  0.5964, -0.0872,  0.1106],
          [-0.7713, -0.6269, -0.2314, -0.9506, -0.8041, -0.8608],
          [-0.7713, -0.6269, -0.2314, -0.9506, -0.8041, -0.8608],
          [ 2.5816,  2.8875,  3.4501,  2.8972,  3.1933,  3.2968],
          [ 2.4638,  2.8057,  3.4626,  2.9959,  3.2028,  3.3292],
          [-0.7791, -0.6359, -0.2053, -0.8944, -0.8273, -0.8935]],
 
         [[-0.7060, -0.5545, -0.1516, -0.8635, -0.7102, -0.7598],
          [-1.2414, -0.8510,  0.0852, -0.1340, -0.3746, -0.2518],
          [-0.7156, -0.5646, -0.1252, -0.8056, -0.7327, -0.7913],
          [-0.7060, -0.5545, -0.1516, -0.8635, -0.7102, -0.7598],
          [ 2.5889,  3.0387,  3.8890,  3.6040,  3.7347,  3.9497],
          [ 2.2757,  2.8038,  3.8429,  3.7315,  3

#### RMSNorm M2

In [102]:
rms2_m2= RMSNorm(embed_dim)
rms2_m2.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [103]:
x_norm2 = rms2_m2(x)
x_norm2.shape, x_norm2

(torch.Size([2, 9, 6]),
 tensor([[[-1.0349, -0.8412, -0.3105, -1.2755, -1.0789, -1.1550],
          [-1.0448, -0.8528, -0.2753, -1.1994, -1.1095, -1.1982],
          [ 0.8268,  0.9315,  1.1278,  0.9591,  1.0436,  1.0805],
          [-1.8889, -1.1889,  0.5953,  0.7932, -0.1159,  0.1471],
          [-1.0349, -0.8412, -0.3105, -1.2755, -1.0789, -1.1550],
          [-1.0349, -0.8412, -0.3105, -1.2755, -1.0789, -1.1550],
          [ 0.8423,  0.9421,  1.1256,  0.9452,  1.0418,  1.0756],
          [ 0.8047,  0.9164,  1.1309,  0.9785,  1.0460,  1.0873],
          [-1.0448, -0.8528, -0.2753, -1.1994, -1.1095, -1.1982]],
 
         [[-1.0611, -0.8333, -0.2279, -1.2978, -1.0674, -1.1419],
          [-1.9254, -1.3198,  0.1321, -0.2079, -0.5810, -0.3906],
          [-1.0751, -0.8482, -0.1881, -1.2102, -1.1008, -1.1887],
          [-1.0611, -0.8333, -0.2279, -1.2978, -1.0674, -1.1419],
          [ 0.7392,  0.8676,  1.1104,  1.0290,  1.0664,  1.1277],
          [ 0.6605,  0.8138,  1.1155,  1.0831,  1

#### SwiGLU

**Gate**

In [104]:
wg_m = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wg_m.weight, 0.5)
wg_m.weight

Parameter containing:
tensor([[0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000, 0.5000, 0.5000]], requires_grad=True)

In [105]:
xwg = wg_m(x_norm2)
xwg.shape, xwg

(torch.Size([2, 9, 8]),
 tensor([[[-2.8480, -2.8480, -2.8480, -2.8480, -2.8480, -2.8480, -2.8480,
           -2.8480],
          [-2.8400, -2.8400, -2.8400, -2.8400, -2.8400, -2.8400, -2.8400,
           -2.8400],
          [ 2.9847,  2.9847,  2.9847,  2.9847,  2.9847,  2.9847,  2.9847,
            2.9847],
          [-0.8290, -0.8290, -0.8290, -0.8290, -0.8290, -0.8290, -0.8290,
           -0.8290],
          [-2.8480, -2.8480, -2.8480, -2.8480, -2.8480, -2.8480, -2.8480,
           -2.8480],
          [-2.8480, -2.8480, -2.8480, -2.8480, -2.8480, -2.8480, -2.8480,
           -2.8480],
          [ 2.9864,  2.9864,  2.9864,  2.9864,  2.9864,  2.9864,  2.9864,
            2.9864],
          [ 2.9819,  2.9819,  2.9819,  2.9819,  2.9819,  2.9819,  2.9819,
            2.9819],
          [-2.8400, -2.8400, -2.8400, -2.8400, -2.8400, -2.8400, -2.8400,
           -2.8400]],
 
         [[-2.8147, -2.8147, -2.8147, -2.8147, -2.8147, -2.8147, -2.8147,
           -2.8147],
          [-2.1463, -2.

**SiLU Non-Linearity**

In [106]:
xwg = F.silu(xwg)
xwg.shape, xwg

(torch.Size([2, 9, 8]),
 tensor([[[-0.1560, -0.1560, -0.1560, -0.1560, -0.1560, -0.1560, -0.1560,
           -0.1560],
          [-0.1568, -0.1568, -0.1568, -0.1568, -0.1568, -0.1568, -0.1568,
           -0.1568],
          [ 2.8411,  2.8411,  2.8411,  2.8411,  2.8411,  2.8411,  2.8411,
            2.8411],
          [-0.2519, -0.2519, -0.2519, -0.2519, -0.2519, -0.2519, -0.2519,
           -0.2519],
          [-0.1560, -0.1560, -0.1560, -0.1560, -0.1560, -0.1560, -0.1560,
           -0.1560],
          [-0.1560, -0.1560, -0.1560, -0.1560, -0.1560, -0.1560, -0.1560,
           -0.1560],
          [ 2.8429,  2.8429,  2.8429,  2.8429,  2.8429,  2.8429,  2.8429,
            2.8429],
          [ 2.8380,  2.8380,  2.8380,  2.8380,  2.8380,  2.8380,  2.8380,
            2.8380],
          [-0.1568, -0.1568, -0.1568, -0.1568, -0.1568, -0.1568, -0.1568,
           -0.1568]],
 
         [[-0.1591, -0.1591, -0.1591, -0.1591, -0.1591, -0.1591, -0.1591,
           -0.1591],
          [-0.2247, -0.

**Weighted Input** 

In [107]:
wu_m = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wu_m.weight, -0.1)
wu_m.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [108]:
xwu = wu_m(x_norm2)
xwu.shape, xwu

(torch.Size([2, 9, 8]),
 tensor([[[ 0.5696,  0.5696,  0.5696,  0.5696,  0.5696,  0.5696,  0.5696,
            0.5696],
          [ 0.5680,  0.5680,  0.5680,  0.5680,  0.5680,  0.5680,  0.5680,
            0.5680],
          [-0.5969, -0.5969, -0.5969, -0.5969, -0.5969, -0.5969, -0.5969,
           -0.5969],
          [ 0.1658,  0.1658,  0.1658,  0.1658,  0.1658,  0.1658,  0.1658,
            0.1658],
          [ 0.5696,  0.5696,  0.5696,  0.5696,  0.5696,  0.5696,  0.5696,
            0.5696],
          [ 0.5696,  0.5696,  0.5696,  0.5696,  0.5696,  0.5696,  0.5696,
            0.5696],
          [-0.5973, -0.5973, -0.5973, -0.5973, -0.5973, -0.5973, -0.5973,
           -0.5973],
          [-0.5964, -0.5964, -0.5964, -0.5964, -0.5964, -0.5964, -0.5964,
           -0.5964],
          [ 0.5680,  0.5680,  0.5680,  0.5680,  0.5680,  0.5680,  0.5680,
            0.5680]],
 
         [[ 0.5629,  0.5629,  0.5629,  0.5629,  0.5629,  0.5629,  0.5629,
            0.5629],
          [ 0.4293,  0.

**Apply Gate**

In [109]:
xw = xwg * xwu
xw.shape, xw

(torch.Size([2, 9, 8]),
 tensor([[[-0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889,
           -0.0889],
          [-0.0890, -0.0890, -0.0890, -0.0890, -0.0890, -0.0890, -0.0890,
           -0.0890],
          [-1.6960, -1.6960, -1.6960, -1.6960, -1.6960, -1.6960, -1.6960,
           -1.6960],
          [-0.0418, -0.0418, -0.0418, -0.0418, -0.0418, -0.0418, -0.0418,
           -0.0418],
          [-0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889,
           -0.0889],
          [-0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889, -0.0889,
           -0.0889],
          [-1.6980, -1.6980, -1.6980, -1.6980, -1.6980, -1.6980, -1.6980,
           -1.6980],
          [-1.6925, -1.6925, -1.6925, -1.6925, -1.6925, -1.6925, -1.6925,
           -1.6925],
          [-0.0890, -0.0890, -0.0890, -0.0890, -0.0890, -0.0890, -0.0890,
           -0.0890]],
 
         [[-0.0896, -0.0896, -0.0896, -0.0896, -0.0896, -0.0896, -0.0896,
           -0.0896],
          [-0.0964, -0.

**Project Down** 

In [110]:
wd_m = nn.Linear(hidden_dim, embed_dim, bias=False)
nn.init.constant_(wd_m.weight, 0.33)
wd_m.weight

Parameter containing:
tensor([[0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300]],
       requires_grad=True)

In [111]:
xswig = wd_m(xw)
xswig.shape, xswig

(torch.Size([2, 9, 6]),
 tensor([[[-0.2346, -0.2346, -0.2346, -0.2346, -0.2346, -0.2346],
          [-0.2351, -0.2351, -0.2351, -0.2351, -0.2351, -0.2351],
          [-4.4774, -4.4774, -4.4774, -4.4774, -4.4774, -4.4774],
          [-0.1103, -0.1103, -0.1103, -0.1103, -0.1103, -0.1103],
          [-0.2346, -0.2346, -0.2346, -0.2346, -0.2346, -0.2346],
          [-0.2346, -0.2346, -0.2346, -0.2346, -0.2346, -0.2346],
          [-4.4827, -4.4827, -4.4827, -4.4827, -4.4827, -4.4827],
          [-4.4682, -4.4682, -4.4682, -4.4682, -4.4682, -4.4682],
          [-0.2351, -0.2351, -0.2351, -0.2351, -0.2351, -0.2351]],
 
         [[-0.2365, -0.2365, -0.2365, -0.2365, -0.2365, -0.2365],
          [-0.2546, -0.2546, -0.2546, -0.2546, -0.2546, -0.2546],
          [-0.2370, -0.2370, -0.2370, -0.2370, -0.2370, -0.2370],
          [-0.2365, -0.2365, -0.2365, -0.2365, -0.2365, -0.2365],
          [-4.4307, -4.4307, -4.4307, -4.4307, -4.4307, -4.4307],
          [-4.3671, -4.3671, -4.3671, -4.3671, -4

#### Residual Connection

In [112]:
x = x + xswig
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-1.0059e+00, -8.6156e-01, -4.6607e-01, -1.1852e+00, -1.0387e+00,
           -1.0954e+00],
          [-1.0141e+00, -8.7100e-01, -4.4039e-01, -1.1295e+00, -1.0624e+00,
           -1.1286e+00],
          [-1.9436e+00, -1.6228e+00, -1.0213e+00, -1.5385e+00, -1.2793e+00,
           -1.1662e+00],
          [-1.5306e+00, -1.0042e+00,  3.3731e-01,  4.8618e-01, -1.9744e-01,
            3.6864e-04],
          [-1.0059e+00, -8.6156e-01, -4.6607e-01, -1.1852e+00, -1.0387e+00,
           -1.0954e+00],
          [-1.0059e+00, -8.6156e-01, -4.6607e-01, -1.1852e+00, -1.0387e+00,
           -1.0954e+00],
          [-1.9011e+00, -1.5952e+00, -1.0327e+00, -1.5856e+00, -1.2895e+00,
           -1.1859e+00],
          [-2.0044e+00, -1.6625e+00, -1.0057e+00, -1.4723e+00, -1.2655e+00,
           -1.1391e+00],
          [-1.0141e+00, -8.7100e-01, -4.4039e-01, -1.1295e+00, -1.0624e+00,
           -1.1286e+00]],
 
         [[-9.4251e-01, -7.9096e-01, -3.8813e-01, -1.1000e+00, -

### Post Attention Layer Normalization

In [113]:
rmsf_m = RMSNorm(embed_dim)
rmsf_m.weight

Parameter containing:
tensor([1., 1., 1., 1., 1., 1.], requires_grad=True)

In [114]:
x = rmsf_m(x)
x.shape, x

(torch.Size([2, 9, 6]),
 tensor([[[-1.0362e+00, -8.8743e-01, -4.8007e-01, -1.2208e+00, -1.0699e+00,
           -1.1283e+00],
          [-1.0442e+00, -8.9685e-01, -4.5346e-01, -1.1630e+00, -1.0939e+00,
           -1.1621e+00],
          [-1.3298e+00, -1.1103e+00, -6.9873e-01, -1.0526e+00, -8.7524e-01,
           -7.9791e-01],
          [-1.9386e+00, -1.2719e+00,  4.2723e-01,  6.1579e-01, -2.5007e-01,
            4.6691e-04],
          [-1.0362e+00, -8.8743e-01, -4.8007e-01, -1.2208e+00, -1.0699e+00,
           -1.1283e+00],
          [-1.0362e+00, -8.8743e-01, -4.8007e-01, -1.2208e+00, -1.0699e+00,
           -1.1283e+00],
          [-1.3012e+00, -1.0918e+00, -7.0679e-01, -1.0852e+00, -8.8255e-01,
           -8.1168e-01],
          [-1.3691e+00, -1.1356e+00, -6.8694e-01, -1.0057e+00, -8.6439e-01,
           -7.7805e-01],
          [-1.0442e+00, -8.9685e-01, -4.5346e-01, -1.1630e+00, -1.0939e+00,
           -1.1621e+00]],
 
         [[-1.0578e+00, -8.8771e-01, -4.3561e-01, -1.2345e+00, -

### Masked Linear Head
Our masked predictor includes a final linear head as part of its output.  Since our attention already projects our results to the embedding dimension, this acts a simple linear layer that maintains the dimensions, though, in other JEPA variants, this layer may project down to the shared latent dimension. 

To keep this layer simple, we'll add a consistent initialization of weights. 

In [115]:
mask_pred_head = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(mask_pred_head.weight, -0.1)
nn.init.zeros_(mask_pred_head.bias)
mask_pred_head.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000]],
       requires_grad=True)

In [116]:
x = mask_pred_head(x)
x

tensor([[[0.5823, 0.5823, 0.5823, 0.5823, 0.5823, 0.5823],
         [0.5814, 0.5814, 0.5814, 0.5814, 0.5814, 0.5814],
         [0.5865, 0.5865, 0.5865, 0.5865, 0.5865, 0.5865],
         [0.2417, 0.2417, 0.2417, 0.2417, 0.2417, 0.2417],
         [0.5823, 0.5823, 0.5823, 0.5823, 0.5823, 0.5823],
         [0.5823, 0.5823, 0.5823, 0.5823, 0.5823, 0.5823],
         [0.5879, 0.5879, 0.5879, 0.5879, 0.5879, 0.5879],
         [0.5840, 0.5840, 0.5840, 0.5840, 0.5840, 0.5840],
         [0.5814, 0.5814, 0.5814, 0.5814, 0.5814, 0.5814]],

        [[0.5796, 0.5796, 0.5796, 0.5796, 0.5796, 0.5796],
         [0.5078, 0.5078, 0.5078, 0.5078, 0.5078, 0.5078],
         [0.5786, 0.5786, 0.5786, 0.5786, 0.5786, 0.5786],
         [0.5796, 0.5796, 0.5796, 0.5796, 0.5796, 0.5796],
         [0.5342, 0.5342, 0.5342, 0.5342, 0.5342, 0.5342],
         [0.5058, 0.5058, 0.5058, 0.5058, 0.5058, 0.5058],
         [0.5796, 0.5796, 0.5796, 0.5796, 0.5796, 0.5796],
         [0.5686, 0.5686, 0.5686, 0.5686, 0.5686, 0.56

### Predicted Latents vs Context Latents

By layering the extra masked predictor, you can see how the model is able to isolate masked prediction away from cell representation. If we compare latent output of our encoder `context_latent` with the masked predictor output `predicted_latent` you can see how the model can learn to separate specifically masked impact vs the cell representation. In our training loss explainer notebook we'll show how these two representations are used. 

In [117]:
predicted_latent = x
predicted_latent

tensor([[[0.5823, 0.5823, 0.5823, 0.5823, 0.5823, 0.5823],
         [0.5814, 0.5814, 0.5814, 0.5814, 0.5814, 0.5814],
         [0.5865, 0.5865, 0.5865, 0.5865, 0.5865, 0.5865],
         [0.2417, 0.2417, 0.2417, 0.2417, 0.2417, 0.2417],
         [0.5823, 0.5823, 0.5823, 0.5823, 0.5823, 0.5823],
         [0.5823, 0.5823, 0.5823, 0.5823, 0.5823, 0.5823],
         [0.5879, 0.5879, 0.5879, 0.5879, 0.5879, 0.5879],
         [0.5840, 0.5840, 0.5840, 0.5840, 0.5840, 0.5840],
         [0.5814, 0.5814, 0.5814, 0.5814, 0.5814, 0.5814]],

        [[0.5796, 0.5796, 0.5796, 0.5796, 0.5796, 0.5796],
         [0.5078, 0.5078, 0.5078, 0.5078, 0.5078, 0.5078],
         [0.5786, 0.5786, 0.5786, 0.5786, 0.5786, 0.5786],
         [0.5796, 0.5796, 0.5796, 0.5796, 0.5796, 0.5796],
         [0.5342, 0.5342, 0.5342, 0.5342, 0.5342, 0.5342],
         [0.5058, 0.5058, 0.5058, 0.5058, 0.5058, 0.5058],
         [0.5796, 0.5796, 0.5796, 0.5796, 0.5796, 0.5796],
         [0.5686, 0.5686, 0.5686, 0.5686, 0.5686, 0.56

In [118]:
context_latent

tensor([[[-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
         [-1.0146, -1.0168, -0.9563, -0.9045, -1.0415, -1.0579],
         [ 0.8914,  0.9262,  1.0169,  1.1000,  1.0144,  1.0365],
         [-1.7992, -1.4326, -0.4755,  0.3999, -0.5021, -0.2689],
         [-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
         [-1.0065, -1.0074, -0.9820, -0.9602, -1.0178, -1.0247],
         [ 0.9388,  0.9586,  1.0102,  1.0575,  1.0088,  1.0214],
         [ 0.8228,  0.8787,  1.0249,  1.1586,  1.0208,  1.0564],
         [-1.0146, -1.0168, -0.9563, -0.9045, -1.0415, -1.0579]],

        [[-1.0078, -1.0082, -0.9821, -0.9597, -1.0171, -1.0235],
         [-1.5903, -1.3566, -0.8019, -0.2915, -0.7475, -0.5863],
         [-1.0179, -1.0189, -0.9563, -0.9024, -1.0403, -1.0557],
         [-1.0078, -1.0082, -0.9821, -0.9597, -1.0171, -1.0235],
         [ 0.5894,  0.7175,  1.0213,  1.3008,  1.0510,  1.1393],
         [ 0.2983,  0.5068,  1.0016,  1.4569,  1.0501,  1.1940],
         [-1.0078, -1.0

# Latent Representation

We now have a latent representation(s) of our input cell states. These representations can then be used in a number of ways. You'll notice that the shape still contains the batch, our context length which is the number of genes, and the embedding dimensions. Joint-embedding predictive architecture (JEPA) models work within this latent space. During our training loops we build energy based loss functions to help steer and shape this space through backprop updates.